# High_Auc.ipynb
## Leakage-Safe Knowledge-Augmented Multimodal Learning for Tox21

**Research target:** maximize the probability of achieving a valid macro mean ROC-AUC ≥ 0.96 on an untouched scaffold-based test set.

> **Scientific integrity:** 0.96 is a stretch target, not a guaranteed result. This notebook never tunes on the final test set, never imputes missing labels as non-toxic, and never uses target-derived assay outcomes as knowledge-graph inputs. The final score must be reproducible under the frozen protocol.

### Central architecture

`AttentiveFP/MPNN molecular graph + leakage-safe GPS/HGT knowledge graph + ChemBERTa + multi-fingerprint encoder → cross-modal attention → endpoint-conditioned gated fusion → NR/SR hierarchical multitask heads → OOF-optimized endpoint-wise ensemble`

### Execution order

1. Environment and paths  
2. Tox21 audit and chemical curation  
3. Per-task masking and frozen scaffold split  
4. Fingerprints, descriptors, graph and SMILES representations  
5. Leakage audit  
6. Baselines and unimodal branches  
7. Safe ToxKG download/import and KG branches  
8. Cross-modal pretraining and gated fusion  
9. Optuna HPO, hybrid heads and OOF ensemble  
10. One-time final evaluation, XAI, calibration, AD, conformal prediction and reporting

## 0. Run profile and hardware note

- `RESEARCH` mode runs the complete pipeline and is computationally expensive.
- `STANDARD` mode keeps the scientific protocol but uses fewer epochs/trials.
- `QUICK` mode is only for debugging the notebook; its scores are not thesis results.
- Full GPS/HGT + ChemBERTa + multimodal training is best on a 24–48 GB GPU. On a smaller GPU, the notebook automatically enables low-memory settings.

In [ ]:
# ============================
# USER CONFIGURATION
# ============================
EXECUTION_MODE = "RESEARCH"   # "RESEARCH", "STANDARD", or "QUICK"
PROJECT_ROOT_OVERRIDE = None   # Example: r"D:\Drug_Toxicity"; None = auto-detect
COLAB_USE_DRIVE = False        # True হলে Google Drive mount করার চেষ্টা করবে
AUTO_DOWNLOAD_TOX21 = True
AUTO_DOWNLOAD_TOXKG = True

# Expensive branches. Keep True for the complete study.
RUN_BASELINES = True
RUN_DEEPTOX = True
RUN_CAPSNET = True
RUN_MOLECULAR_GNN = True
RUN_CHEMBERTA = True
RUN_KG_MODELS = True
RUN_MULTIMODAL_FUSION = True
RUN_HYBRIDS = True
RUN_HPO = True
RUN_XAI = True
RUN_OPTIONAL_DENSENET = True
RUN_OPTIONAL_3D_EGNN = False   # Enable after the core model is stable and GPU permits
RUN_SELF_SUPERVISED_PRETRAINING = True
RUN_CHEMBERTA_DOMAIN_ADAPTATION = True
RUN_CONFIRMATORY_5FOLD = False    # Enable only after architecture/HPO are frozen and compute permits

# Frozen scientific protocol
TRAIN_FRACTION = 0.75
N_CV_FOLDS = 3
FINAL_CONFIRMATORY_FOLDS = 5
FINAL_SEEDS = [13, 29, 47]
PRIMARY_SEED = 13
TARGET_MEAN_AUC = 0.96

print("গবেষণা কনফিগারেশন লোড হয়েছে।")
print(f"Execution mode: {EXECUTION_MODE}")

## 1. Environment setup

The next cell checks required libraries. Missing packages are installed automatically. Before any installation or model/data download, a **Bangla message** is printed.

In [ ]:
# প্রয়োজনীয় package যাচাই ও install
import importlib.util, subprocess, sys, os, platform

PACKAGE_MAP = {
    "numpy": "numpy",
    "pandas": "pandas",
    "scipy": "scipy",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "tqdm": "tqdm",
    "yaml": "pyyaml",
    "joblib": "joblib",
    "pyarrow": "pyarrow",
    "rdkit": "rdkit",
    "torch": "torch",
    "torchvision": "torchvision",
    "torch_geometric": "torch-geometric",
    "transformers": "transformers>=4.45",
    "accelerate": "accelerate",
    "optuna": "optuna",
    "xgboost": "xgboost",
    "lightgbm": "lightgbm",
    "huggingface_hub": "huggingface_hub",
    "captum": "captum",
    "shap": "shap",
    "pubchempy": "pubchempy",
}

missing = [pip_name for module, pip_name in PACKAGE_MAP.items()
           if importlib.util.find_spec(module) is None]

if missing:
    print("প্রয়োজনীয় কিছু Python package পাওয়া যায়নি। এখন সেগুলো install করা হচ্ছে...")
    print("Install list:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("সব প্রয়োজনীয় package install সম্পন্ন হয়েছে। Kernel restart চাইলে একবার restart করে পুনরায় Run All দিন।")
else:
    print("সব প্রয়োজনীয় package আগে থেকেই install করা আছে।")

In [ ]:
# Core imports
import os, re, gc, json, math, time, random, hashlib, warnings, shutil, platform, sys
from dataclasses import dataclass, asdict
from pathlib import Path
from collections import defaultdict, Counter
from typing import Dict, List, Tuple, Optional, Iterable, Any

import numpy as np
import pandas as pd
from scipy import sparse
from scipy.optimize import minimize
from scipy.special import expit, logit

import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML, clear_output

from sklearn.base import clone
from sklearn.feature_selection import VarianceThreshold
from sklearn.decomposition import TruncatedSVD, PCA
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import (
    roc_auc_score, average_precision_score, accuracy_score,
    balanced_accuracy_score, f1_score, matthews_corrcoef,
    confusion_matrix, roc_curve, precision_recall_curve,
    brier_score_loss
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import GroupKFold
from sklearn.neighbors import NearestNeighbors
import joblib

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, TensorDataset, WeightedRandomSampler

from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import AllChem, MACCSkeys, Descriptors, Crippen, Lipinski, rdMolDescriptors
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem import Draw

warnings.filterwarnings("ignore")
RDLogger.DisableLog("rdApp.warning")
RDLogger.DisableLog("rdApp.error")
sns.set_context("notebook")
sns.set_style("whitegrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

print("সব core library সফলভাবে import হয়েছে।")
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)

In [ ]:
# Clean notebook output style
HTML("""
<style>
.jp-RenderedHTMLCommon h1, .jp-RenderedHTMLCommon h2, .jp-RenderedHTMLCommon h3 {
  color: #173B7A; font-weight: 700;
}
.output_area pre {font-size: 13px; line-height: 1.35;}
.dataframe {font-size: 12px;}
.research-card {padding:14px 18px;border-left:5px solid #2D6CDF;background:#F4F7FF;border-radius:8px;margin:8px 0;}
.success-card {padding:12px 16px;border-left:5px solid #1A9B76;background:#EFFBF7;border-radius:8px;}
.warning-card {padding:12px 16px;border-left:5px solid #E6A700;background:#FFF9E8;border-radius:8px;}
</style>
""")

## 2. Cross-platform path resolver

Supported layouts:

- Windows: `D:\Drug_Toxicity\datasetsaw	ox21.csv`
- Your earlier layout: `D:\Drug_Toxicity\datasets	ox21.csv`
- Colab/Drive: `/content/drive/MyDrive/Drug_Toxicity/...`
- Portable fallback: the current folder or its parent

In [ ]:
# Optional Google Drive mount
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and COLAB_USE_DRIVE:
    print("Google Drive mount করা হচ্ছে...")
    from google.colab import drive
    drive.mount("/content/drive")
    print("Google Drive mount সম্পন্ন হয়েছে।")

In [ ]:
def resolve_project_root(override=None) -> Path:
    candidates = []
    if override:
        candidates.append(Path(override))
    if IN_COLAB:
        candidates += [
            Path("/content/drive/MyDrive/Drug_Toxicity"),
            Path("/content/Drug_Toxicity"),
        ]
    candidates += [
        Path(r"D:\Drug_Toxicity"),
        Path.cwd(),
        Path.cwd().parent,
    ]
    for p in candidates:
        try:
            if p.exists() and (p / "datasets").exists():
                return p.resolve()
        except Exception:
            pass
    root = Path.cwd() / "Drug_Toxicity"
    root.mkdir(parents=True, exist_ok=True)
    return root.resolve()

ROOT = resolve_project_root(PROJECT_ROOT_OVERRIDE)
DIRS = {
    "datasets": ROOT / "datasets",
    "raw": ROOT / "datasets" / "raw",
    "processed": ROOT / "datasets" / "processed",
    "splits": ROOT / "datasets" / "splits",
    "cache": ROOT / "datasets" / "cache",
    "external": ROOT / "datasets" / "external",
    "kg_raw": ROOT / "knowledge_graph" / "raw",
    "kg_processed": ROOT / "knowledge_graph" / "processed" / "toxkg_safe_v1",
    "kg_provenance": ROOT / "knowledge_graph" / "provenance",
    "notebooks": ROOT / "notebooks",
    "models": ROOT / "models",
    "outputs": ROOT / "outputs",
    "figures": ROOT / "figures",
    "logs": ROOT / "logs",
    "configs": ROOT / "configs",
    "requirements": ROOT / "requirements",
    "reports": ROOT / "reports",
}
for p in DIRS.values():
    p.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
print("Output folder:", DIRS["outputs"])

## 3. Reproducibility and compute profile

All final candidates use deterministic split manifests. Final candidates are repeated across three seeds. Mixed precision and low-memory settings are selected from available hardware.

In [ ]:
def seed_everything(seed: int = 13):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(PRIMARY_SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU_GB = 0.0
if torch.cuda.is_available():
    GPU_GB = torch.cuda.get_device_properties(0).total_memory / (1024**3)
LOW_MEMORY = (DEVICE.type == "cpu") or (GPU_GB < 16)
USE_AMP = DEVICE.type == "cuda"

MODE_CONFIG = {
    "QUICK": dict(epochs=4, patience=2, batch_size=32, hpo_trials=2, seeds=[PRIMARY_SEED], max_samples=1200),
    "STANDARD": dict(epochs=35, patience=7, batch_size=32 if LOW_MEMORY else 64, hpo_trials=12, seeds=[PRIMARY_SEED], max_samples=None),
    "RESEARCH": dict(epochs=120, patience=12, batch_size=16 if LOW_MEMORY else 64, hpo_trials=30, seeds=FINAL_SEEDS, max_samples=None),
}
CFG = MODE_CONFIG[EXECUTION_MODE]

print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)} ({GPU_GB:.1f} GB)")
print("Low-memory mode:", LOW_MEMORY)
print("Training configuration:", CFG)

In [ ]:
# Save the frozen run configuration
RUN_CONFIG = {
    "execution_mode": EXECUTION_MODE,
    "train_fraction": TRAIN_FRACTION,
    "n_cv_folds": N_CV_FOLDS,
    "primary_seed": PRIMARY_SEED,
    "target_mean_auc": TARGET_MEAN_AUC,
    "device": str(DEVICE),
    "low_memory": LOW_MEMORY,
    "mode_config": CFG,
}
with open(DIRS["configs"] / "high_auc_run_config.json", "w", encoding="utf-8") as f:
    json.dump(RUN_CONFIG, f, indent=2, ensure_ascii=False)
print("Frozen configuration saved.")

# Part I — Data audit, chemical curation and frozen protocol

## 4. Locate or download Tox21

If `tox21.csv` is not found, the notebook downloads the public DeepChem Tox21 CSV archive and stores it under `datasets/raw/`. A Bangla message is shown before downloading.

In [ ]:
def find_tox21_file() -> Optional[Path]:
    candidates = [
        DIRS["raw"] / "tox21.csv",
        DIRS["datasets"] / "tox21.csv",
        ROOT / "tox21.csv",
        Path.cwd() / "tox21.csv",
    ]
    for p in candidates:
        if p.exists():
            return p
    return None

TOX21_PATH = find_tox21_file()
if TOX21_PATH is None and AUTO_DOWNLOAD_TOX21:
    print("tox21.csv পাওয়া যায়নি। এখন public Tox21 dataset download করা হচ্ছে...")
    import urllib.request, gzip
    url = "https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/tox21.csv.gz"
    gz_path = DIRS["raw"] / "tox21.csv.gz"
    try:
        urllib.request.urlretrieve(url, gz_path)
        with gzip.open(gz_path, "rb") as fin, open(DIRS["raw"] / "tox21.csv", "wb") as fout:
            shutil.copyfileobj(fin, fout)
        TOX21_PATH = DIRS["raw"] / "tox21.csv"
        print("Tox21 dataset download ও extract সম্পন্ন হয়েছে।")
    except Exception as e:
        print("স্বয়ংক্রিয় download ব্যর্থ হয়েছে:", e)

if TOX21_PATH is None:
    raise FileNotFoundError(
        "tox21.csv পাওয়া যায়নি। ফাইলটি D:/Drug_Toxicity/datasets/raw/ অথবা datasets/ ফোল্ডারে রাখুন।"
    )
print("Dataset path:", TOX21_PATH)

In [ ]:
# Load raw data
raw_df = pd.read_csv(TOX21_PATH)
SMILES_COL = next((c for c in raw_df.columns if c.lower() == "smiles"), None)
ID_COL = next((c for c in raw_df.columns if c.lower() in {"mol_id", "molid", "id", "compound_id"}), None)
if SMILES_COL is None:
    raise ValueError("SMILES column পাওয়া যায়নি।")

NON_TARGET = {SMILES_COL}
if ID_COL:
    NON_TARGET.add(ID_COL)
TARGET_COLS = [c for c in raw_df.columns if c not in NON_TARGET]
for c in TARGET_COLS:
    raw_df[c] = pd.to_numeric(raw_df[c], errors="coerce")
    raw_df.loc[~raw_df[c].isin([0, 1]), c] = np.nan

print(f"Raw dataset: {raw_df.shape[0]:,} molecules × {raw_df.shape[1]} columns")
print(f"Detected endpoints ({len(TARGET_COLS)}):", TARGET_COLS)
display(raw_df.head(3))

## 5. Initial dataset audit

The audit reports missing labels, toxic/non-toxic counts, positive rate and imbalance ratio for every endpoint. Missing labels are preserved as missing.

In [ ]:
def endpoint_audit(df: pd.DataFrame, targets: List[str]) -> pd.DataFrame:
    rows = []
    for t in targets:
        s = df[t]
        n_pos = int((s == 1).sum())
        n_neg = int((s == 0).sum())
        n_missing = int(s.isna().sum())
        labeled = n_pos + n_neg
        rows.append({
            "Endpoint": t,
            "Labeled": labeled,
            "Toxic (1)": n_pos,
            "Non-Toxic (0)": n_neg,
            "Missing": n_missing,
            "Missing %": 100 * n_missing / len(df),
            "Positive %": 100 * n_pos / max(labeled, 1),
            "Neg:Pos": n_neg / max(n_pos, 1),
        })
    return pd.DataFrame(rows)

raw_audit = endpoint_audit(raw_df, TARGET_COLS)
display(raw_audit.round(3))
raw_audit.to_csv(DIRS["outputs"] / "raw_endpoint_audit.csv", index=False)

In [ ]:
# Dataset challenge visualizations
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

plot_df = raw_audit.sort_values("Missing %")
axes[0].barh(plot_df["Endpoint"], plot_df["Missing %"])
axes[0].axvline(15, linestyle="--", linewidth=1.2, label="15% reference")
axes[0].set_title("Missing Labels per Endpoint")
axes[0].set_xlabel("Missing labels (%)")
axes[0].legend()

plot_df = raw_audit.sort_values("Positive %")
axes[1].barh(plot_df["Endpoint"], plot_df["Positive %"])
axes[1].set_title("Class Imbalance — Toxic Positive Rate")
axes[1].set_xlabel("Toxic positive rate (%)")

plt.suptitle("Tox21 Dataset Challenges", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig(DIRS["figures"] / "01_dataset_challenges.png", dpi=220, bbox_inches="tight")
plt.show()

## 6. RDKit chemical standardization

Pipeline: `SMILES → parse → largest organic fragment → normalize → uncharge → sanitize → canonical SMILES + InChIKey + Bemis–Murcko scaffold`.

All downstream representations are generated from the same standardized molecule identity.

In [ ]:
# Chemical standardization helpers
fragment_chooser = rdMolStandardize.LargestFragmentChooser(preferOrganic=True)
normalizer = rdMolStandardize.Normalizer()
uncharger = rdMolStandardize.Uncharger()


def standardize_smiles(smiles: str) -> Dict[str, Any]:
    out = {
        "original_smiles": smiles,
        "canonical_smiles": None,
        "inchikey": None,
        "scaffold": None,
        "status": "invalid",
        "reason": None,
    }
    try:
        mol = Chem.MolFromSmiles(str(smiles), sanitize=True)
        if mol is None:
            out["reason"] = "MolFromSmiles failed"
            return out
        mol = fragment_chooser.choose(mol)
        mol = normalizer.normalize(mol)
        mol = uncharger.uncharge(mol)
        Chem.SanitizeMol(mol)
        can = Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)
        ik = Chem.MolToInchiKey(mol)
        scaffold = MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=True)
        if scaffold == "":
            scaffold = f"ACYCLIC::{Chem.MolToSmiles(mol, canonical=True)}"
        out.update({
            "canonical_smiles": can,
            "inchikey": ik,
            "scaffold": scaffold,
            "status": "valid",
            "reason": "",
        })
        return out
    except Exception as e:
        out["reason"] = str(e)[:180]
        return out

In [ ]:
# Apply curation and keep a complete audit table
from tqdm.auto import tqdm

tqdm.pandas(desc="Chemical curation")
curation_records = raw_df[SMILES_COL].progress_apply(standardize_smiles).tolist()
curation_audit = pd.DataFrame(curation_records)
curation_audit.insert(0, "original_row_id", np.arange(len(raw_df)))

work_df = pd.concat([raw_df.reset_index(drop=True), curation_audit.drop(columns=["original_smiles"])], axis=1)
invalid_df = work_df[work_df["status"] != "valid"].copy()
valid_df = work_df[work_df["status"] == "valid"].copy()

print(f"Valid molecules: {len(valid_df):,}")
print(f"Invalid molecules removed: {len(invalid_df):,}")
curation_audit.to_csv(DIRS["outputs"] / "chemical_curation_audit.csv", index=False)
invalid_df.to_csv(DIRS["outputs"] / "invalid_smiles_records.csv", index=False)

## 7. Duplicate merge and conflicting-label policy

- Same canonical molecule + consistent labels → merge.
- Same canonical molecule + conflicting label for a specific endpoint → only that endpoint becomes missing.
- Other valid endpoint labels are retained.

In [ ]:
# Vectorized duplicate merge: fast even when most molecules are unique.
grp = valid_df.groupby("canonical_smiles", sort=False)
curated_df = valid_df.drop_duplicates("canonical_smiles", keep="first").copy()
curated_df = curated_df.set_index("canonical_smiles", drop=False)

source_ids = grp["original_row_id"].agg(lambda s: ",".join(map(str, s.tolist())))
dup_counts = grp.size()
curated_df["source_row_ids"] = source_ids.reindex(curated_df.index).values
curated_df["duplicate_count"] = dup_counts.reindex(curated_df.index).values
curated_df["conflicting_endpoints"] = 0

for t in TARGET_COLS:
    stats = grp[t].agg(["min", "max", "count"])
    vals = stats["min"].astype(float)
    vals[stats["count"] == 0] = np.nan
    conflict = (stats["count"] > 0) & (stats["min"] != stats["max"])
    vals[conflict] = np.nan
    curated_df[t] = vals.reindex(curated_df.index).values
    curated_df["conflicting_endpoints"] += conflict.reindex(curated_df.index, fill_value=False).astype(int).values

curated_df = curated_df.reset_index(drop=True)
print(f"After duplicate merge: {len(curated_df):,} unique standardized molecules")
print("Conflicting endpoint labels set to missing:", int(curated_df["conflicting_endpoints"].sum()))

audit_curated = endpoint_audit(curated_df, TARGET_COLS)
display(audit_curated.round(3))

In [ ]:
# Save curated dataset and identity manifest
CURATED_CSV = DIRS["processed"] / "tox21_curated.csv"
CURATED_PARQUET = DIRS["processed"] / "tox21_curated.parquet"
curated_df.to_csv(CURATED_CSV, index=False)
try:
    curated_df.to_parquet(CURATED_PARQUET, index=False)
except Exception as e:
    print("Parquet save skipped:", e)
print("Curated dataset saved:", CURATED_CSV)

## 8. Chemical-space descriptors and visual audit

In [ ]:
CHEM_SPACE_DESC = {
    "MW": Descriptors.MolWt,
    "LogP": Crippen.MolLogP,
    "TPSA": rdMolDescriptors.CalcTPSA,
    "HBD": Lipinski.NumHDonors,
    "HBA": Lipinski.NumHAcceptors,
    "RotBonds": Lipinski.NumRotatableBonds,
    "Rings": Lipinski.RingCount,
    "AromaticRings": Lipinski.NumAromaticRings,
    "HeavyAtoms": Lipinski.HeavyAtomCount,
    "FractionCSP3": rdMolDescriptors.CalcFractionCSP3,
}

def compute_chemical_space_row(smiles):
    mol = Chem.MolFromSmiles(smiles)
    d = {k: float(fn(mol)) for k, fn in CHEM_SPACE_DESC.items()}
    d["SMILES_Length"] = len(smiles)
    return d

chem_space = pd.DataFrame([compute_chemical_space_row(s) for s in tqdm(curated_df["canonical_smiles"], desc="Chemical space")])
chem_space.to_csv(DIRS["outputs"] / "chemical_space_descriptors.csv", index=False)
display(chem_space.describe().T.round(3))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, col in zip(axes.ravel(), ["MW", "LogP", "TPSA", "Rings", "SMILES_Length", "FractionCSP3"]):
    ax.hist(chem_space[col].dropna(), bins=35, alpha=0.85)
    ax.set_title(col)
    ax.set_ylabel("Molecules")
plt.suptitle("Curated Tox21 Chemical Space", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig(DIRS["figures"] / "02_chemical_space.png", dpi=220, bbox_inches="tight")
plt.show()

## 9. Frozen 75/25 scaffold split

The final test set is frozen before model development. A greedy multi-start group assignment balances scaffold groups, total size and endpoint-positive support without breaking scaffold separation.

In [ ]:
def label_stats_for_indices(y: np.ndarray, idx: np.ndarray):
    yy = y[idx]
    valid = ~np.isnan(yy)
    pos = np.nansum(yy, axis=0)
    labeled = valid.sum(axis=0)
    return pos.astype(float), labeled.astype(float)


def scaffold_holdout_assignment(df, targets, test_fraction=0.25, seed=13, n_starts=200):
    y = df[targets].to_numpy(dtype=float)
    groups = df["scaffold"].astype(str).values
    unique_groups = np.unique(groups)
    group_indices = {g: np.where(groups == g)[0] for g in unique_groups}
    total_pos, total_lab = label_stats_for_indices(y, np.arange(len(df)))
    target_n = len(df) * test_fraction
    target_pos = total_pos * test_fraction
    target_lab = total_lab * test_fraction

    best = None
    rng = np.random.default_rng(seed)
    for start in range(n_starts):
        order = list(unique_groups)
        rng.shuffle(order)
        order.sort(key=lambda g: len(group_indices[g]), reverse=True)
        test_groups = []
        n_test = 0
        pos_test = np.zeros(len(targets))
        lab_test = np.zeros(len(targets))
        for g in order:
            idx = group_indices[g]
            gp, gl = label_stats_for_indices(y, idx)
            def cost(n, p, l):
                size_cost = abs(n - target_n) / max(target_n, 1)
                pos_cost = np.mean(np.abs(p - target_pos) / np.maximum(target_pos, 3))
                lab_cost = np.mean(np.abs(l - target_lab) / np.maximum(target_lab, 10))
                over = max(0, n - target_n * 1.08) / max(target_n, 1)
                return 0.45 * size_cost + 0.40 * pos_cost + 0.15 * lab_cost + 2.0 * over
            c_add = cost(n_test + len(idx), pos_test + gp, lab_test + gl)
            c_skip = cost(n_test, pos_test, lab_test)
            if c_add <= c_skip or n_test < target_n * 0.88:
                test_groups.append(g)
                n_test += len(idx)
                pos_test += gp
                lab_test += gl
        final_cost = cost(n_test, pos_test, lab_test)
        if best is None or final_cost < best[0]:
            best = (final_cost, set(test_groups))
    test_group_set = best[1]
    split = np.where(np.isin(groups, list(test_group_set)), "test", "train")
    return split, best[0]

In [ ]:
SPLIT_FILE = DIRS["splits"] / "scaffold_split_v1.csv"

if SPLIT_FILE.exists():
    print("আগে freeze করা scaffold split পাওয়া গেছে; সেটিই ব্যবহার করা হচ্ছে।")
    split_manifest = pd.read_csv(SPLIT_FILE)
    merge_cols = ["canonical_smiles", "split"]
    curated_df = curated_df.drop(columns=["split"], errors="ignore").merge(split_manifest[merge_cols], on="canonical_smiles", how="left")
    if curated_df["split"].isna().any():
        raise ValueError("Curated dataset ও split manifest mismatch. Version audit করুন।")
else:
    print("নতুন leakage-safe 75/25 scaffold split তৈরি ও freeze করা হচ্ছে...")
    split_labels, split_cost = scaffold_holdout_assignment(
        curated_df, TARGET_COLS, test_fraction=1-TRAIN_FRACTION,
        seed=PRIMARY_SEED, n_starts=40 if EXECUTION_MODE == "QUICK" else 300
    )
    curated_df["split"] = split_labels
    split_manifest = curated_df[["canonical_smiles", "inchikey", "scaffold", "split"]].copy()
    split_manifest.to_csv(SPLIT_FILE, index=False)
    print(f"Scaffold split freeze সম্পন্ন হয়েছে। Assignment cost={split_cost:.4f}")

train_idx = np.where(curated_df["split"].values == "train")[0]
test_idx = np.where(curated_df["split"].values == "test")[0]
print(f"বাংলা বার্তা: Dataset ভাগ করা হয়েছে — Train = {len(train_idx):,} ({len(train_idx)/len(curated_df):.1%}), Test = {len(test_idx):,} ({len(test_idx)/len(curated_df):.1%})")

In [ ]:
# Split integrity checks
train_scaffolds = set(curated_df.loc[train_idx, "scaffold"])
test_scaffolds = set(curated_df.loc[test_idx, "scaffold"])
train_ik = set(curated_df.loc[train_idx, "inchikey"])
test_ik = set(curated_df.loc[test_idx, "inchikey"])

assert train_scaffolds.isdisjoint(test_scaffolds), "Scaffold leakage detected!"
assert train_ik.isdisjoint(test_ik), "Molecule identity leakage detected!"

split_quality = pd.concat([
    endpoint_audit(curated_df.iloc[train_idx], TARGET_COLS).assign(Split="Train"),
    endpoint_audit(curated_df.iloc[test_idx], TARGET_COLS).assign(Split="Test"),
])
display(split_quality[["Split","Endpoint","Labeled","Toxic (1)","Non-Toxic (0)","Missing","Positive %"]].round(2))
print("Leakage check passed: zero molecule and scaffold overlap.")

## 10. Scaffold-aware 3-fold CV inside the training pool

The final test is never used for hyperparameter search, early stopping, thresholds or ensemble weights.

In [ ]:
def balanced_group_folds(df_train, targets, n_splits=3, seed=13, n_starts=120):
    """Greedy multi-start scaffold-group assignment with global balance objective."""
    y = df_train[targets].to_numpy(float)
    groups = df_train["scaffold"].astype(str).values
    unique_groups = np.unique(groups)
    gidx = {g: np.where(groups == g)[0] for g in unique_groups}
    gp = {}; gl = {}
    for g, idx in gidx.items():
        gp[g], gl[g] = label_stats_for_indices(y, idx)
    total_pos, total_lab = label_stats_for_indices(y, np.arange(len(df_train)))
    target_n = len(df_train) / n_splits
    target_pos = total_pos / n_splits
    target_lab = total_lab / n_splits
    rng = np.random.default_rng(seed)
    best = None

    def global_cost(ns, ps, ls):
        size_term = np.std(ns / max(target_n, 1))
        pos_term = np.mean(np.std(ps / np.maximum(target_pos, 3), axis=0))
        lab_term = np.mean(np.std(ls / np.maximum(target_lab, 10), axis=0))
        empty_penalty = 5.0 * np.sum(ns == 0)
        return 0.45 * size_term + 0.40 * pos_term + 0.15 * lab_term + empty_penalty

    for _ in range(n_starts):
        order = list(unique_groups)
        rng.shuffle(order)
        # Large and positive-rich groups are assigned first; random jitter changes ties across starts.
        order.sort(key=lambda g: (len(gidx[g]) + 0.25 * gp[g].sum() + rng.random()*1e-3), reverse=True)
        fold_groups = [set() for _ in range(n_splits)]
        ns = np.zeros(n_splits, dtype=float)
        ps = np.zeros((n_splits, len(targets)), dtype=float)
        ls = np.zeros((n_splits, len(targets)), dtype=float)

        # Seed every fold to guarantee non-empty folds.
        for k, g in enumerate(order[:n_splits]):
            fold_groups[k].add(g); ns[k] += len(gidx[g]); ps[k] += gp[g]; ls[k] += gl[g]

        for g in order[n_splits:]:
            costs = []
            for k in range(n_splits):
                ns2, ps2, ls2 = ns.copy(), ps.copy(), ls.copy()
                ns2[k] += len(gidx[g]); ps2[k] += gp[g]; ls2[k] += gl[g]
                # Soft overflow penalty discourages a single giant fold.
                overflow = max(0.0, ns2[k] - 1.12*target_n) / max(target_n, 1)
                costs.append(global_cost(ns2, ps2, ls2) + 0.5*overflow)
            min_cost = min(costs)
            candidates = [k for k,cost in enumerate(costs) if abs(cost-min_cost) < 1e-12]
            k = int(rng.choice(candidates))
            fold_groups[k].add(g); ns[k] += len(gidx[g]); ps[k] += gp[g]; ls[k] += gl[g]

        cost = global_cost(ns, ps, ls)
        if best is None or cost < best[0]:
            best = (cost, fold_groups, ns.copy())

    fold_id = np.full(len(df_train), -1, dtype=int)
    for k, gs in enumerate(best[1]):
        fold_id[np.isin(groups, list(gs))] = k
    if (fold_id < 0).any() or len(np.unique(fold_id)) != n_splits:
        raise RuntimeError("Balanced scaffold fold assignment failed.")
    return fold_id, best[0]

train_df = curated_df.iloc[train_idx].reset_index().rename(columns={"index":"global_index"})
CV_FILE = DIRS["splits"] / "scaffold_cv3_v1.csv"
if CV_FILE.exists():
    cv_manifest = pd.read_csv(CV_FILE)
    train_df = train_df.merge(cv_manifest[["canonical_smiles","cv_fold"]], on="canonical_smiles", how="left")
    if train_df["cv_fold"].nunique() != N_CV_FOLDS:
        print("পুরনো CV manifest invalid/অসম্পূর্ণ; নতুন balanced manifest তৈরি হচ্ছে...")
        CV_FILE.unlink()
        fold_ids, fold_cost = balanced_group_folds(train_df, TARGET_COLS, n_splits=N_CV_FOLDS, seed=PRIMARY_SEED,
                                                   n_starts=20 if EXECUTION_MODE=="QUICK" else 160)
        train_df["cv_fold"] = fold_ids
        train_df[["canonical_smiles","inchikey","scaffold","cv_fold"]].to_csv(CV_FILE,index=False)
else:
    fold_ids, fold_cost = balanced_group_folds(
        train_df, TARGET_COLS, n_splits=N_CV_FOLDS, seed=PRIMARY_SEED,
        n_starts=20 if EXECUTION_MODE == "QUICK" else 160
    )
    train_df["cv_fold"] = fold_ids
    train_df[["canonical_smiles","inchikey","scaffold","cv_fold"]].to_csv(CV_FILE, index=False)
    print(f"3-fold scaffold-aware CV manifest saved. Cost={fold_cost:.4f}")

assert train_df["cv_fold"].notna().all()
assert train_df["cv_fold"].nunique() == N_CV_FOLDS
for k in range(N_CV_FOLDS):
    v = train_df[train_df.cv_fold == k]
    print(f"Fold {k}: {len(v):,} molecules, {v.scaffold.nunique():,} scaffolds")

In [ ]:
# Per-task labels and masks
Y = curated_df[TARGET_COLS].to_numpy(dtype=np.float32)
M = (~np.isnan(Y)).astype(np.float32)
Y_FILLED = np.nan_to_num(Y, nan=0.0).astype(np.float32)

assert np.all((Y_FILLED == 0) | (Y_FILLED == 1))
assert np.all((M == 0) | (M == 1))
print("Per-task mask তৈরি হয়েছে। Missing label কোনো অবস্থাতেই 0/1 হিসেবে impute করা হয়নি।")
print(f"Observed label cells: {int(M.sum()):,}; missing cells: {int((1-M).sum()):,}")

# Part II — Molecular representations and leakage-safe utilities

## 11. Multi-fingerprint and descriptor generation

Fingerprints:

- ECFP4 bit vector
- Morgan count vector
- MACCS keys
- Atom-Pair
- RDKit fingerprint
- Optional topological torsion

Descriptors are median-imputed, scaled and optionally reduced **inside each training fold only**.

In [ ]:
# Fingerprint and descriptor definitions
FP_SIZES = {
    "ecfp4": 2048,
    "morgan_count": 2048,
    "maccs": 167,
    "atom_pair": 2048,
    "rdkit": 2048,
    "torsion": 2048,
}

DESC_FUNCS = {
    "MolWt": Descriptors.MolWt,
    "MolLogP": Crippen.MolLogP,
    "TPSA": rdMolDescriptors.CalcTPSA,
    "HBD": Lipinski.NumHDonors,
    "HBA": Lipinski.NumHAcceptors,
    "RotatableBonds": Lipinski.NumRotatableBonds,
    "RingCount": Lipinski.RingCount,
    "AromaticRings": Lipinski.NumAromaticRings,
    "AliphaticRings": Lipinski.NumAliphaticRings,
    "HeavyAtoms": Lipinski.HeavyAtomCount,
    "FractionCSP3": rdMolDescriptors.CalcFractionCSP3,
    "FormalCharge": Chem.GetFormalCharge,
    "NumHeteroatoms": Lipinski.NumHeteroatoms,
    "NumValenceElectrons": Descriptors.NumValenceElectrons,
    "MolMR": Crippen.MolMR,
    "LabuteASA": rdMolDescriptors.CalcLabuteASA,
    "NHOHCount": Lipinski.NHOHCount,
    "NOCount": Lipinski.NOCount,
    "NumRadicalElectrons": Descriptors.NumRadicalElectrons,
}

def bitvect_to_array(fp, n_bits):
    arr = np.zeros((n_bits,), dtype=np.float32)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

def sparse_count_to_hashed_array(sv, n_bits):
    arr = np.zeros(n_bits, dtype=np.float32)
    for idx, val in sv.GetNonzeroElements().items():
        arr[idx % n_bits] += float(val)
    return arr

def molecule_features(smiles: str):
    mol = Chem.MolFromSmiles(smiles)
    ecfp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=FP_SIZES["ecfp4"], useChirality=True)
    mcount = AllChem.GetMorganFingerprint(mol, radius=2, useChirality=True)
    maccs = MACCSkeys.GenMACCSKeys(mol)
    ap = rdMolDescriptors.GetHashedAtomPairFingerprintAsBitVect(mol, nBits=FP_SIZES["atom_pair"])
    rdk = Chem.RDKFingerprint(mol, fpSize=FP_SIZES["rdkit"], minPath=1, maxPath=7)
    torsion = rdMolDescriptors.GetHashedTopologicalTorsionFingerprintAsBitVect(mol, nBits=FP_SIZES["torsion"])
    fps = np.concatenate([
        bitvect_to_array(ecfp, FP_SIZES["ecfp4"]),
        sparse_count_to_hashed_array(mcount, FP_SIZES["morgan_count"]),
        bitvect_to_array(maccs, FP_SIZES["maccs"]),
        bitvect_to_array(ap, FP_SIZES["atom_pair"]),
        bitvect_to_array(rdk, FP_SIZES["rdkit"]),
        bitvect_to_array(torsion, FP_SIZES["torsion"]),
    ]).astype(np.float32)
    desc = np.array([float(fn(mol)) for fn in DESC_FUNCS.values()], dtype=np.float32)
    return fps, desc

In [ ]:
FEATURE_CACHE = DIRS["cache"] / "multifingerprint_descriptors_v1.npz"
if FEATURE_CACHE.exists():
    print("Cached fingerprint/descriptor features load করা হচ্ছে...")
    cache = np.load(FEATURE_CACHE)
    X_FP = cache["X_FP"]
    X_DESC = cache["X_DESC"]
else:
    print("Fingerprint ও molecular descriptor generate করা হচ্ছে...")
    fp_rows, desc_rows = [], []
    for smi in tqdm(curated_df["canonical_smiles"], desc="Molecular features"):
        fp, ds = molecule_features(smi)
        fp_rows.append(fp); desc_rows.append(ds)
    X_FP = np.stack(fp_rows).astype(np.float32)
    X_DESC = np.stack(desc_rows).astype(np.float32)
    np.savez_compressed(FEATURE_CACHE, X_FP=X_FP, X_DESC=X_DESC)
    print("Feature generation complete এবং cache save হয়েছে।")

print("Fingerprint matrix:", X_FP.shape)
print("Descriptor matrix:", X_DESC.shape)

In [ ]:
class FoldLocalFeatureProcessor:
    """Fits every preprocessing step on fold-train only."""
    def __init__(self, max_svd=512, desc_pca_var=0.99, random_state=13):
        self.fp_var = VarianceThreshold(0.0)
        self.fp_svd = None
        self.desc_imputer = SimpleImputer(strategy="median")
        self.desc_scaler = RobustScaler()
        self.desc_pca = None
        self.max_svd = max_svd
        self.desc_pca_var = desc_pca_var
        self.random_state = random_state

    def fit(self, fp, desc):
        fpv = self.fp_var.fit_transform(fp)
        n_comp = min(self.max_svd, max(2, fpv.shape[0]-1), max(2, fpv.shape[1]-1))
        self.fp_svd = TruncatedSVD(n_components=n_comp, random_state=self.random_state)
        self.fp_svd.fit(fpv)
        ds = self.desc_imputer.fit_transform(desc)
        ds = self.desc_scaler.fit_transform(ds)
        self.desc_pca = PCA(n_components=self.desc_pca_var, random_state=self.random_state)
        self.desc_pca.fit(ds)
        return self

    def transform(self, fp, desc):
        fpz = self.fp_svd.transform(self.fp_var.transform(fp))
        dsz = self.desc_pca.transform(self.desc_scaler.transform(self.desc_imputer.transform(desc)))
        return np.concatenate([fpz, dsz], axis=1).astype(np.float32)

    def fit_transform(self, fp, desc):
        return self.fit(fp, desc).transform(fp, desc)

## 12. Molecular graph representation

Atom features include atomic number, degree, formal charge, hybridization, aromaticity, chirality, hydrogen count, valence, ring membership, mass and donor/acceptor flags. Bond features include bond type, order, conjugation, ring status and stereo.

In [ ]:
# PyG imports are isolated so the rest of the notebook remains usable if PyG needs a kernel restart.
try:
    from torch_geometric.data import Data, HeteroData
    from torch_geometric.loader import DataLoader as PyGDataLoader
    from torch_geometric.nn.models import AttentiveFP
    from torch_geometric.nn import NNConv, Set2Set, GATv2Conv, global_mean_pool, HGTConv, RGCNConv, GPSConv, GINEConv
    PYG_AVAILABLE = True
    print("PyTorch Geometric ready.")
except Exception as e:
    PYG_AVAILABLE = False
    print("PyTorch Geometric import হয়নি:", e)

In [ ]:
ATOM_LIST = [1, 5, 6, 7, 8, 9, 14, 15, 16, 17, 35, 53]
HYBRID_LIST = [Chem.rdchem.HybridizationType.SP, Chem.rdchem.HybridizationType.SP2,
               Chem.rdchem.HybridizationType.SP3, Chem.rdchem.HybridizationType.SP3D,
               Chem.rdchem.HybridizationType.SP3D2]
BOND_TYPES = [Chem.rdchem.BondType.SINGLE, Chem.rdchem.BondType.DOUBLE,
              Chem.rdchem.BondType.TRIPLE, Chem.rdchem.BondType.AROMATIC]

def one_hot(value, choices):
    return [float(value == c) for c in choices] + [float(value not in choices)]

def atom_features(atom):
    return np.array(
        one_hot(atom.GetAtomicNum(), ATOM_LIST)
        + one_hot(atom.GetDegree(), [0,1,2,3,4,5])
        + one_hot(atom.GetFormalCharge(), [-2,-1,0,1,2])
        + one_hot(atom.GetHybridization(), HYBRID_LIST)
        + [
            float(atom.GetIsAromatic()), float(atom.HasProp("_ChiralityPossible")),
            float(atom.GetTotalNumHs()) / 4.0, float(atom.GetTotalValence()) / 8.0,
            float(atom.IsInRing()), float(atom.GetMass()) / 200.0,
            float(atom.GetAtomicNum() in {7,8,9,15,16}),
        ], dtype=np.float32)

def bond_features(bond):
    return np.array(
        one_hot(bond.GetBondType(), BOND_TYPES)
        + [float(bond.GetBondTypeAsDouble())/3.0,
           float(bond.GetIsConjugated()), float(bond.IsInRing())]
        + one_hot(bond.GetStereo(), [Chem.rdchem.BondStereo.STEREONONE,
                                     Chem.rdchem.BondStereo.STEREOZ,
                                     Chem.rdchem.BondStereo.STEREOE]),
        dtype=np.float32
    )

def smiles_to_pyg(smiles, y_row=None, mask_row=None, idx=None):
    mol = Chem.MolFromSmiles(smiles)
    x = torch.tensor(np.stack([atom_features(a) for a in mol.GetAtoms()]), dtype=torch.float32)
    edges, eattrs = [], []
    for b in mol.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        bf = bond_features(b)
        edges.extend([[i,j],[j,i]]); eattrs.extend([bf,bf])
    if edges:
        edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
        edge_attr = torch.tensor(np.stack(eattrs), dtype=torch.float32)
    else:
        edge_index = torch.zeros((2,0), dtype=torch.long)
        edge_attr = torch.zeros((0, len(bond_features(Chem.MolFromSmiles("CC").GetBondWithIdx(0)))), dtype=torch.float32)
    data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr)
    if y_row is not None:
        data.y = torch.tensor(y_row, dtype=torch.float32).view(1,-1)
        data.mask = torch.tensor(mask_row, dtype=torch.float32).view(1,-1)
    if idx is not None:
        data.sample_idx = torch.tensor([idx], dtype=torch.long)
    return data

In [ ]:
GRAPH_CACHE = DIRS["cache"] / "pyg_molecular_graphs_v1.pt"
if PYG_AVAILABLE:
    if GRAPH_CACHE.exists():
        print("Cached molecular graphs load করা হচ্ছে...")
        MOLECULAR_GRAPHS = torch.load(GRAPH_CACHE, weights_only=False)
    else:
        print("SMILES থেকে molecular graph তৈরি করা হচ্ছে...")
        MOLECULAR_GRAPHS = [
            smiles_to_pyg(s, Y_FILLED[i], M[i], i)
            for i, s in enumerate(tqdm(curated_df["canonical_smiles"], desc="Molecular graphs"))
        ]
        torch.save(MOLECULAR_GRAPHS, GRAPH_CACHE)
        print("Molecular graph cache save হয়েছে।")
else:
    MOLECULAR_GRAPHS = None

## 13. SMILES augmentation

Canonical SMILES are used for deterministic validation/test. Randomized SMILES are generated only during training. Test-time augmentation uses a small fixed set generated without labels.

In [ ]:
def randomized_smiles(canonical_smiles: str, n: int = 1, seed: int = 13) -> List[str]:
    mol = Chem.MolFromSmiles(canonical_smiles)
    if mol is None:
        return [canonical_smiles] * n
    out = []
    # RDKit randomization is stochastic; seed the global RNG for reproducibility.
    state = random.getstate(); random.seed(seed)
    for _ in range(n):
        out.append(Chem.MolToSmiles(mol, canonical=False, doRandom=True, isomericSmiles=True))
    random.setstate(state)
    return out

example_smi = curated_df["canonical_smiles"].iloc[0]
print("Canonical:", example_smi)
print("Randomized views:", randomized_smiles(example_smi, 3, PRIMARY_SEED))

## 14. Metric, threshold, calibration and reporting utilities

Headline metrics: macro mean ROC-AUC and Accuracy. Diagnostic metrics: PR-AUC, Balanced Accuracy, F1, MCC, sensitivity and specificity.

In [ ]:
def safe_auc(y, p):
    y = np.asarray(y); p = np.asarray(p)
    return roc_auc_score(y, p) if len(np.unique(y)) == 2 else np.nan

def safe_ap(y, p):
    y = np.asarray(y); p = np.asarray(p)
    return average_precision_score(y, p) if len(np.unique(y)) == 2 else np.nan

def specificity_score(y, pred):
    cm = confusion_matrix(y, pred, labels=[0,1])
    tn, fp, fn, tp = cm.ravel()
    return tn / max(tn+fp, 1)

def sensitivity_score(y, pred):
    cm = confusion_matrix(y, pred, labels=[0,1])
    tn, fp, fn, tp = cm.ravel()
    return tp / max(tp+fn, 1)

def optimize_threshold(y, p, criterion="balanced_accuracy"):
    grid = np.unique(np.r_[np.linspace(0.02,0.98,97), p])
    best_t, best_s = 0.5, -np.inf
    for t in grid:
        pred = (p >= t).astype(int)
        if criterion == "accuracy":
            s = accuracy_score(y,pred)
        elif criterion == "youden":
            s = sensitivity_score(y,pred) + specificity_score(y,pred) - 1
        else:
            s = balanced_accuracy_score(y,pred)
        if s > best_s:
            best_t, best_s = float(t), float(s)
    return best_t

def evaluate_multitask(y_true, mask, prob, thresholds=None, endpoints=None):
    endpoints = endpoints or [f"Task_{i}" for i in range(y_true.shape[1])]
    rows=[]
    if thresholds is None:
        thresholds = np.full(y_true.shape[1], 0.5)
    for j,t in enumerate(endpoints):
        idx = mask[:,j].astype(bool)
        y = y_true[idx,j].astype(int); p = prob[idx,j]
        if len(y)==0:
            continue
        pred = (p >= thresholds[j]).astype(int)
        rows.append({
            "Endpoint":t, "N":len(y), "Positive":int(y.sum()), "Negative":int((1-y).sum()),
            "AUC":safe_auc(y,p), "PR_AUC":safe_ap(y,p),
            "Accuracy":accuracy_score(y,pred),
            "Balanced_Accuracy":balanced_accuracy_score(y,pred) if len(np.unique(y))==2 else np.nan,
            "F1":f1_score(y,pred,zero_division=0),
            "MCC":matthews_corrcoef(y,pred) if len(np.unique(pred))>1 else 0.0,
            "Sensitivity":sensitivity_score(y,pred), "Specificity":specificity_score(y,pred),
            "Threshold":thresholds[j]
        })
    return pd.DataFrame(rows)

def macro_summary(per_task_df):
    cols = ["AUC","PR_AUC","Accuracy","Balanced_Accuracy","F1","MCC","Sensitivity","Specificity"]
    return {f"Mean_{c}": float(per_task_df[c].mean(skipna=True)) for c in cols}

In [ ]:
def thresholds_from_oof(y_true, mask, prob, criterion="balanced_accuracy"):
    th = np.full(y_true.shape[1], 0.5, dtype=float)
    for j in range(y_true.shape[1]):
        idx = mask[:,j].astype(bool)
        y, p = y_true[idx,j].astype(int), prob[idx,j]
        if len(np.unique(y)) == 2:
            th[j] = optimize_threshold(y,p,criterion)
    return th

def bootstrap_macro_auc(y_true, mask, prob, n_boot=1000, seed=13):
    rng = np.random.default_rng(seed)
    n = len(y_true); vals=[]
    for _ in range(n_boot):
        idx = rng.integers(0,n,n)
        scores=[]
        for j in range(y_true.shape[1]):
            valid = mask[idx,j].astype(bool)
            y = y_true[idx,j][valid]; p = prob[idx,j][valid]
            if len(np.unique(y)) == 2:
                scores.append(roc_auc_score(y,p))
        if scores:
            vals.append(np.mean(scores))
    return np.percentile(vals,[2.5,50,97.5]) if vals else [np.nan]*3

def expected_calibration_error(y, p, n_bins=10):
    bins=np.linspace(0,1,n_bins+1); ece=0.0
    for lo,hi in zip(bins[:-1],bins[1:]):
        m=(p>=lo)&(p<(hi if hi<1 else hi+1e-9))
        if m.any():
            ece += m.mean()*abs(y[m].mean()-p[m].mean())
    return float(ece)

In [ ]:
# Artifact registry — every branch writes OOF/test probabilities and embeddings here.
ARTIFACT_DIR = DIRS["outputs"] / "model_artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)
MODEL_REGISTRY = {}

def save_model_artifact(name, oof_prob, test_prob, oof_embed=None, test_embed=None, metadata=None):
    path = ARTIFACT_DIR / f"{name}.npz"
    payload = {"oof_prob":oof_prob.astype(np.float32), "test_prob":test_prob.astype(np.float32)}
    if oof_embed is not None: payload["oof_embed"] = oof_embed.astype(np.float32)
    if test_embed is not None: payload["test_embed"] = test_embed.astype(np.float32)
    np.savez_compressed(path, **payload)
    meta = metadata or {}
    with open(ARTIFACT_DIR / f"{name}.json","w",encoding="utf-8") as f:
        json.dump(meta,f,indent=2,ensure_ascii=False)
    MODEL_REGISTRY[name] = {"path":str(path), "metadata":meta}
    print(f"Saved artifact: {name}")

def load_model_artifact(name):
    return dict(np.load(ARTIFACT_DIR / f"{name}.npz"))

def discover_artifacts():
    return sorted(p.stem for p in ARTIFACT_DIR.glob("*.npz"))

## 15. Masked class-balanced focal/asymmetric losses

Class statistics are computed from each fold's training portion only. The notebook compares masked class-balanced focal BCE and masked asymmetric loss.

In [ ]:
def effective_num_weights(y, m, beta=0.999):
    pos = (y*m).sum(dim=0).clamp_min(1.0)
    neg = ((1-y)*m).sum(dim=0).clamp_min(1.0)
    w_pos = (1-beta) / torch.clamp(1-torch.pow(torch.tensor(beta,device=y.device),pos), min=1e-8)
    w_neg = (1-beta) / torch.clamp(1-torch.pow(torch.tensor(beta,device=y.device),neg), min=1e-8)
    w_pos = (w_pos / torch.clamp(w_pos.mean(),min=1e-8)).clamp(0.2, 5.0)
    w_neg = (w_neg / torch.clamp(w_neg.mean(),min=1e-8)).clamp(0.2, 5.0)
    return w_pos.detach(), w_neg.detach()

class MaskedClassBalancedFocalLoss(nn.Module):
    def __init__(self, w_pos, w_neg, gamma=2.0):
        super().__init__(); self.register_buffer("w_pos",w_pos); self.register_buffer("w_neg",w_neg); self.gamma=gamma
    def forward(self, logits, y, mask):
        p=torch.sigmoid(logits)
        bce=nn.functional.binary_cross_entropy_with_logits(logits,y,reduction="none")
        pt=y*p+(1-y)*(1-p)
        w=y*self.w_pos+(1-y)*self.w_neg
        loss=w*torch.pow(1-pt,self.gamma)*bce*mask
        per_task=loss.sum(0)/mask.sum(0).clamp_min(1.0)
        return per_task.mean()

class MaskedAsymmetricLoss(nn.Module):
    def __init__(self, w_pos, w_neg, gamma_pos=1.0, gamma_neg=4.0, clip=0.05):
        super().__init__(); self.register_buffer("w_pos",w_pos); self.register_buffer("w_neg",w_neg)
        self.gp=gamma_pos; self.gn=gamma_neg; self.clip=clip
    def forward(self, logits,y,mask):
        p=torch.sigmoid(logits); pn=(1-p+self.clip).clamp(max=1)
        pos=-y*torch.log(p.clamp_min(1e-8))*torch.pow(1-p,self.gp)*self.w_pos
        neg=-(1-y)*torch.log(pn.clamp_min(1e-8))*torch.pow(p,self.gn)*self.w_neg
        loss=(pos+neg)*mask
        return (loss.sum(0)/mask.sum(0).clamp_min(1)).mean()

# Part III — Baselines and unimodal branch models

The classical models are retained only as reproducible thesis baselines and hybrid heads. They are not the central contribution.

## 16. Classical same-split baselines: RF, SVM-RBF and XGBoost

In [ ]:
def probability_like(model, X):
    """Return ranking-preserving probabilities/scores for AUC without forcing expensive SVC Platt CV."""
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    score = model.decision_function(X)
    return expit(np.clip(score, -30, 30))

def fit_predict_per_task(model_factory, Xtr, Xva, Xte, ytr, mtr, yva, mva):
    n_tasks=ytr.shape[1]
    pva=np.full((len(Xva),n_tasks),np.nan,dtype=np.float32)
    pte=np.full((len(Xte),n_tasks),np.nan,dtype=np.float32)
    for j in range(n_tasks):
        idx=mtr[:,j].astype(bool)
        if len(np.unique(ytr[idx,j]))<2: continue
        model=model_factory(j,ytr[idx,j])
        model.fit(Xtr[idx],ytr[idx,j].astype(int))
        pva[:,j]=probability_like(model, Xva)
        pte[:,j]=probability_like(model, Xte)
    return pva,pte

def run_classical_cv(name, model_factory):
    oof=np.full((len(train_df),len(TARGET_COLS)),np.nan,dtype=np.float32)
    test_fold=[]
    for fold in range(N_CV_FOLDS):
        tr_local=np.where(train_df.cv_fold.values!=fold)[0]
        va_local=np.where(train_df.cv_fold.values==fold)[0]
        tr_global=train_df.global_index.values[tr_local]
        va_global=train_df.global_index.values[va_local]
        proc=FoldLocalFeatureProcessor(max_svd=192 if LOW_MEMORY else 384, random_state=PRIMARY_SEED+fold)
        Xtr=proc.fit_transform(X_FP[tr_global],X_DESC[tr_global])
        Xva=proc.transform(X_FP[va_global],X_DESC[va_global])
        Xte=proc.transform(X_FP[test_idx],X_DESC[test_idx])
        pva,pte=fit_predict_per_task(model_factory,Xtr,Xva,Xte,Y_FILLED[tr_global],M[tr_global],Y_FILLED[va_global],M[va_global])
        oof[va_local]=pva; test_fold.append(pte)
        print(f"{name}: fold {fold+1}/{N_CV_FOLDS} complete")
    test_prob=np.nanmean(np.stack(test_fold),axis=0)
    save_model_artifact(name,oof,test_prob,metadata={"type":"classical","cv":N_CV_FOLDS})
    return oof,test_prob

In [ ]:
if RUN_BASELINES:
    def rf_factory(j,y):
        return RandomForestClassifier(
            n_estimators=250 if EXECUTION_MODE!="QUICK" else 50,
            max_features="sqrt", min_samples_leaf=2,
            class_weight="balanced_subsample", n_jobs=-1, random_state=PRIMARY_SEED+j
        )
    RF_OOF, RF_TEST = run_classical_cv("RandomForest", rf_factory)
else:
    print("Baseline models disabled by configuration.")

In [ ]:
if RUN_BASELINES:
    def svm_factory(j,y):
        return SVC(C=4.0, gamma="scale", kernel="rbf", probability=False,
                   class_weight="balanced", cache_size=4096, random_state=PRIMARY_SEED+j)
    SVM_OOF, SVM_TEST = run_classical_cv("SVM_RBF", svm_factory)

In [ ]:
if RUN_BASELINES:
    try:
        from xgboost import XGBClassifier
        def xgb_factory(j,y):
            npos=max(1,(y==1).sum()); nneg=max(1,(y==0).sum())
            return XGBClassifier(
                n_estimators=500 if EXECUTION_MODE!="QUICK" else 60,
                max_depth=5, learning_rate=0.035, subsample=0.85, colsample_bytree=0.8,
                reg_lambda=2.0, reg_alpha=0.05, scale_pos_weight=nneg/npos,
                eval_metric="logloss", n_jobs=-1, random_state=PRIMARY_SEED+j
            )
        XGB_OOF, XGB_TEST = run_classical_cv("XGBoost", xgb_factory)
    except Exception as e:
        print("XGBoost skipped:",e)

## 17. Reusable neural training utilities

Training includes AMP, gradient clipping, early stopping, cosine scheduling and best-checkpoint restore. Loss and metrics are fully masked per endpoint.

In [ ]:
@dataclass
class TrainHistory:
    train_loss: List[float]
    val_auc: List[float]
    best_epoch: int
    best_auc: float


def mean_masked_auc(y, m, p):
    vals=[]
    for j in range(y.shape[1]):
        idx=m[:,j].astype(bool)
        if idx.sum()>1 and len(np.unique(y[idx,j]))==2:
            vals.append(roc_auc_score(y[idx,j],p[idx,j]))
    return float(np.mean(vals)) if vals else np.nan


def train_array_model(model, Xtr, ytr, mtr, Xva, yva, mva, seed=13,
                      epochs=None, batch_size=None, lr=3e-4, weight_decay=1e-5,
                      loss_name="focal", patience=None):
    seed_everything(seed)
    epochs=epochs or CFG["epochs"]; batch_size=batch_size or CFG["batch_size"]; patience=patience or CFG["patience"]
    model=model.to(DEVICE)
    train_ds=TensorDataset(torch.tensor(Xtr),torch.tensor(ytr),torch.tensor(mtr))
    train_loader=DataLoader(train_ds,batch_size=batch_size,shuffle=True,num_workers=0,pin_memory=DEVICE.type=="cuda")
    wpos,wneg=effective_num_weights(torch.tensor(ytr,device=DEVICE),torch.tensor(mtr,device=DEVICE))
    criterion=MaskedAsymmetricLoss(wpos,wneg) if loss_name=="asymmetric" else MaskedClassBalancedFocalLoss(wpos,wneg,gamma=2.0)
    opt=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=weight_decay)
    sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=max(epochs,1))
    scaler=torch.amp.GradScaler("cuda",enabled=USE_AMP)
    best_auc=-np.inf; best_state=None; wait=0; history=TrainHistory([],[],0,-np.inf)
    Xva_t=torch.tensor(Xva,device=DEVICE)
    for epoch in range(epochs):
        model.train(); losses=[]
        for xb,yb,mb in train_loader:
            xb,yb,mb=xb.to(DEVICE),yb.to(DEVICE),mb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda",enabled=USE_AMP):
                out=model(xb)
                logits=out[0] if isinstance(out,tuple) else out
                loss=criterion(logits,yb,mb)
            scaler.scale(loss).backward()
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(),1.0)
            scaler.step(opt); scaler.update(); losses.append(loss.item())
        sched.step()
        model.eval()
        with torch.no_grad(), torch.amp.autocast("cuda",enabled=USE_AMP):
            out=model(Xva_t); logits=out[0] if isinstance(out,tuple) else out
            p=torch.sigmoid(logits).float().cpu().numpy()
        auc=mean_masked_auc(yva,mva,p)
        history.train_loss.append(float(np.mean(losses))); history.val_auc.append(auc)
        if auc>best_auc+1e-5:
            best_auc=auc; best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; wait=0; history.best_epoch=epoch; history.best_auc=auc
        else:
            wait+=1
        if wait>=patience: break
    if best_state is not None: model.load_state_dict(best_state)
    return model,history

def predict_array_model(model,X,batch_size=512):
    model.eval(); probs=[]; embeds=[]
    with torch.no_grad():
        for i in range(0,len(X),batch_size):
            xb=torch.tensor(X[i:i+batch_size],device=DEVICE)
            with torch.amp.autocast("cuda",enabled=USE_AMP):
                out=model(xb)
            if isinstance(out,tuple): logits,z=out
            else: logits,z=out,None
            probs.append(torch.sigmoid(logits).float().cpu().numpy())
            if z is not None: embeds.append(z.float().cpu().numpy())
    return np.vstack(probs), (np.vstack(embeds) if embeds else None)

## 18. DeepTox-style multitask DNN

A strong fingerprint/descriptor branch with residual blocks, BatchNorm, LeakyReLU, dropout and 12 sigmoid task heads.

In [ ]:
class ResidualMLPBlock(nn.Module):
    def __init__(self,d,dropout=0.25):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(d,d),nn.BatchNorm1d(d),nn.LeakyReLU(0.1),nn.Dropout(dropout),nn.Linear(d,d),nn.BatchNorm1d(d))
        self.act=nn.LeakyReLU(0.1)
    def forward(self,x): return self.act(x+self.net(x))

class DeepToxDNN(nn.Module):
    def __init__(self,input_dim,n_tasks,hidden=512,embed_dim=256,dropout=0.28):
        super().__init__()
        self.stem=nn.Sequential(nn.Linear(input_dim,hidden),nn.BatchNorm1d(hidden),nn.LeakyReLU(0.1),nn.Dropout(dropout))
        self.blocks=nn.Sequential(ResidualMLPBlock(hidden,dropout),ResidualMLPBlock(hidden,dropout))
        self.embed=nn.Sequential(nn.Linear(hidden,embed_dim),nn.LeakyReLU(0.1),nn.Dropout(dropout/2))
        self.head=nn.Linear(embed_dim,n_tasks)
    def forward(self,x):
        z=self.embed(self.blocks(self.stem(x)))
        return self.head(z),z

In [ ]:
def run_array_branch_cv(name, model_builder, loss_name="focal"):
    oof=np.full((len(train_df),len(TARGET_COLS)),np.nan,np.float32); test_probs=[]
    oof_emb=None; test_embs=[]
    for fold in range(N_CV_FOLDS):
        trl=np.where(train_df.cv_fold.values!=fold)[0]; val=np.where(train_df.cv_fold.values==fold)[0]
        trg=train_df.global_index.values[trl]; vag=train_df.global_index.values[val]
        proc=FoldLocalFeatureProcessor(max_svd=256 if LOW_MEMORY else 512,random_state=PRIMARY_SEED+fold)
        Xtr=proc.fit_transform(X_FP[trg],X_DESC[trg]); Xva=proc.transform(X_FP[vag],X_DESC[vag]); Xte=proc.transform(X_FP[test_idx],X_DESC[test_idx])
        model=model_builder(Xtr.shape[1])
        model,h=train_array_model(model,Xtr,Y_FILLED[trg],M[trg],Xva,Y_FILLED[vag],M[vag],seed=PRIMARY_SEED+fold,loss_name=loss_name)
        pva,zva=predict_array_model(model,Xva); pte,zte=predict_array_model(model,Xte)
        torch.save({"state_dict":model.state_dict(),"fold":fold,"processor":proc,"metadata":{"name":name}},DIRS["models"]/f"{name}_fold{fold}.pt")
        oof[val]=pva; test_probs.append(pte)
        if zva is not None:
            if oof_emb is None: oof_emb=np.zeros((len(train_df),zva.shape[1]),np.float32)
            oof_emb[val]=zva; test_embs.append(zte)
        print(f"{name}: fold {fold+1}/{N_CV_FOLDS}, best AUC={h.best_auc:.4f}")
        gc.collect();
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    test_prob=np.mean(test_probs,axis=0); test_embed=np.mean(test_embs,axis=0) if test_embs else None
    save_model_artifact(name,oof,test_prob,oof_emb,test_embed,{"type":"multitask_array","cv":N_CV_FOLDS,"loss":loss_name})
    return oof,test_prob,oof_emb,test_embed

In [ ]:
if RUN_DEEPTOX:
    DEEPTOX_OOF,DEEPTOX_TEST,DEEPTOX_OOF_EMB,DEEPTOX_TEST_EMB=run_array_branch_cv(
        "DeepTox_DNN",
        lambda d: DeepToxDNN(d,len(TARGET_COLS),hidden=384 if LOW_MEMORY else 512,embed_dim=256),
        loss_name="focal"
    )

## 19. Multitask CapsNet

CapsNet is retained as a rare/minority-pattern specialist. It is kept in the final ensemble only if it improves OOF AUC or minority-class diagnostics without causing AUC collapse.

In [ ]:
class PrimaryCapsules(nn.Module):
    def __init__(self,input_dim,num_caps=16,cap_dim=16):
        super().__init__(); self.num_caps=num_caps; self.cap_dim=cap_dim; self.proj=nn.Linear(input_dim,num_caps*cap_dim)
    def forward(self,x):
        u=self.proj(x).view(-1,self.num_caps,self.cap_dim)
        norm=torch.norm(u,dim=-1,keepdim=True)
        return (norm**2/(1+norm**2))*u/(norm+1e-8)

class TaskCapsules(nn.Module):
    def __init__(self,in_caps,in_dim,n_tasks,out_dim=16,routing=3):
        super().__init__(); self.W=nn.Parameter(torch.randn(1,in_caps,n_tasks,out_dim,in_dim)*0.05); self.routing=routing; self.n_tasks=n_tasks
    def forward(self,u):
        B,I,D=u.shape; uhat=torch.matmul(self.W.expand(B,-1,-1,-1,-1),u[:, :, None, :, None]).squeeze(-1)
        b=torch.zeros(B,I,self.n_tasks,device=u.device)
        for r in range(self.routing):
            c=torch.softmax(b,dim=2)
            s=(c[...,None]*uhat).sum(dim=1)
            n=torch.norm(s,dim=-1,keepdim=True)
            v=(n**2/(1+n**2))*s/(n+1e-8)
            if r<self.routing-1: b=b+(uhat*v[:,None,:,:]).sum(-1)
        return v

class MultitaskCapsNet(nn.Module):
    def __init__(self,input_dim,n_tasks,hidden=384,embed_dim=256,dropout=0.25):
        super().__init__(); self.encoder=nn.Sequential(nn.Linear(input_dim,hidden),nn.BatchNorm1d(hidden),nn.LeakyReLU(0.1),nn.Dropout(dropout))
        self.primary=PrimaryCapsules(hidden,16,16); self.task_caps=TaskCapsules(16,16,n_tasks,16,3)
        self.embed=nn.Linear(n_tasks*16,embed_dim); self.head=nn.Linear(16,1)
    def forward(self,x):
        tc=self.task_caps(self.primary(self.encoder(x)))
        logits=self.head(tc).squeeze(-1)
        z=torch.tanh(self.embed(tc.flatten(1)))
        return logits,z

In [ ]:
if RUN_CAPSNET:
    CAPS_OOF,CAPS_TEST,CAPS_OOF_EMB,CAPS_TEST_EMB=run_array_branch_cv(
        "CapsNet",
        lambda d: MultitaskCapsNet(d,len(TARGET_COLS),hidden=320 if LOW_MEMORY else 448,embed_dim=256),
        loss_name="asymmetric"
    )

## 20. AttentiveFP and MPNN molecular graph branches

These branches learn local atom–bond structure and provide atom-level explanation candidates.

In [ ]:
if PYG_AVAILABLE:
    NODE_DIM=MOLECULAR_GRAPHS[0].x.shape[1]
    EDGE_DIM=MOLECULAR_GRAPHS[0].edge_attr.shape[1]

    class AttentiveFPModel(nn.Module):
        def __init__(self,n_tasks,hidden=256,layers=4,dropout=0.25):
            super().__init__(); self.model=AttentiveFP(NODE_DIM,hidden,EDGE_DIM,hidden,n_tasks,layers,2,dropout)
            self.embed_proj=nn.Sequential(nn.Linear(n_tasks,256),nn.LeakyReLU(0.1))
        def forward(self,data):
            logits=self.model(data.x,data.edge_index,data.edge_attr,data.batch)
            return logits,self.embed_proj(logits)

    class MPNNModel(nn.Module):
        def __init__(self,n_tasks,hidden=256,steps=4,dropout=0.25):
            super().__init__(); self.node=nn.Linear(NODE_DIM,hidden)
            edge_net=nn.Sequential(nn.Linear(EDGE_DIM,hidden),nn.ReLU(),nn.Linear(hidden,hidden*hidden))
            self.conv=NNConv(hidden,hidden,edge_net,aggr="mean"); self.gru=nn.GRU(hidden,hidden)
            self.steps=steps; self.pool=Set2Set(hidden,processing_steps=3); self.drop=nn.Dropout(dropout)
            self.embed=nn.Linear(2*hidden,256); self.head=nn.Linear(256,n_tasks)
        def forward(self,data):
            x=torch.relu(self.node(data.x)); h=x.unsqueeze(0)
            for _ in range(self.steps):
                m=torch.relu(self.conv(x,data.edge_index,data.edge_attr)); x,h=self.gru(m.unsqueeze(0),h); x=x.squeeze(0)
            g=self.pool(x,data.batch); z=torch.relu(self.embed(self.drop(g))); return self.head(z),z

In [ ]:
def pretrain_graph_contrastive(model, loader, epochs=2, lr=2e-4, edge_drop=0.12):
    """Label-free graph/KG contrastive pretraining on fold-train graphs only."""
    if not RUN_SELF_SUPERVISED_PRETRAINING or epochs <= 0:
        return model
    model=model.to(DEVICE);opt=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=1e-5)
    model.train()
    for _ in range(epochs):
        for b in loader:
            b=b.to(DEVICE);b2=b.clone()
            if b2.edge_index.size(1)>2:
                keep=torch.rand(b2.edge_index.size(1),device=b2.edge_index.device)>edge_drop
                if keep.sum()<2:keep[:2]=True
                b2.edge_index=b2.edge_index[:,keep]
                if hasattr(b2,"edge_attr") and b2.edge_attr is not None:b2.edge_attr=b2.edge_attr[keep]
                if hasattr(b2,"edge_type") and b2.edge_type is not None:b2.edge_type=b2.edge_type[keep]
            if hasattr(b2,"x") and b2.x is not None:
                feat_keep=(torch.rand_like(b2.x)>.08).float();b2.x=b2.x*feat_keep
            opt.zero_grad(set_to_none=True);_,z1=model(b);_,z2=model(b2)
            z1=nn.functional.normalize(z1,dim=-1);z2=nn.functional.normalize(z2,dim=-1)
            logits=z1@z2.T/.12;labels=torch.arange(len(z1),device=z1.device)
            loss=.5*(nn.functional.cross_entropy(logits,labels)+nn.functional.cross_entropy(logits.T,labels))
            loss.backward();nn.utils.clip_grad_norm_(model.parameters(),1.0);opt.step()
    return model

def graph_predict(model,loader):
    model.eval(); ps=[]; zs=[]; idxs=[]
    with torch.no_grad():
        for batch in loader:
            batch=batch.to(DEVICE); logits,z=model(batch)
            ps.append(torch.sigmoid(logits).cpu().numpy()); zs.append(z.cpu().numpy()); idxs.append(batch.sample_idx.view(-1).cpu().numpy())
    return np.concatenate(idxs),np.vstack(ps),np.vstack(zs)

def train_graph_model(model,tr_graphs,va_graphs,seed=13,loss_name="focal"):
    seed_everything(seed); model=model.to(DEVICE)
    tr_loader=PyGDataLoader(tr_graphs,batch_size=CFG["batch_size"],shuffle=True)
    pre_epochs = 1 if EXECUTION_MODE == "QUICK" else (3 if EXECUTION_MODE == "STANDARD" else 8)
    model = pretrain_graph_contrastive(model, tr_loader, epochs=pre_epochs)
    va_loader=PyGDataLoader(va_graphs,batch_size=CFG["batch_size"],shuffle=False)
    yt=np.vstack([g.y.numpy() for g in tr_graphs]); mt=np.vstack([g.mask.numpy() for g in tr_graphs])
    wpos,wneg=effective_num_weights(torch.tensor(yt,device=DEVICE),torch.tensor(mt,device=DEVICE))
    criterion=MaskedAsymmetricLoss(wpos,wneg) if loss_name=="asymmetric" else MaskedClassBalancedFocalLoss(wpos,wneg)
    opt=torch.optim.AdamW(model.parameters(),lr=3e-4,weight_decay=1e-5)
    sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=CFG["epochs"])
    scaler=torch.amp.GradScaler("cuda",enabled=USE_AMP); best=-np.inf; state=None; wait=0
    for epoch in range(CFG["epochs"]):
        model.train()
        for b in tr_loader:
            b=b.to(DEVICE); opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda",enabled=USE_AMP):
                logits,_=model(b); loss=criterion(logits,b.y,b.mask)
            scaler.scale(loss).backward(); scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(),1.0); scaler.step(opt); scaler.update()
        sched.step(); idx,p,_=graph_predict(model,va_loader)
        # Order by sample_idx to align with va_graphs
        order=np.argsort(idx); p=p[order]
        yv=np.vstack([g.y.numpy() for g in sorted(va_graphs,key=lambda x:int(x.sample_idx.item()))])
        mv=np.vstack([g.mask.numpy() for g in sorted(va_graphs,key=lambda x:int(x.sample_idx.item()))])
        auc=mean_masked_auc(yv,mv,p)
        if auc>best+1e-5: best=auc; state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; wait=0
        else: wait+=1
        if wait>=CFG["patience"]: break
    if state: model.load_state_dict(state)
    return model,best

def run_graph_branch_cv(name,builder,loss_name="focal"):
    oof=np.full((len(train_df),len(TARGET_COLS)),np.nan,np.float32); oof_z=np.zeros((len(train_df),256),np.float32); test_ps=[];test_zs=[]
    for fold in range(N_CV_FOLDS):
        trl=np.where(train_df.cv_fold.values!=fold)[0]; val=np.where(train_df.cv_fold.values==fold)[0]
        trg=train_df.global_index.values[trl]; vag=train_df.global_index.values[val]
        model,best=train_graph_model(builder(),[MOLECULAR_GRAPHS[i] for i in trg],[MOLECULAR_GRAPHS[i] for i in vag],PRIMARY_SEED+fold,loss_name)
        torch.save({"state_dict":model.state_dict(),"fold":fold,"name":name},DIRS["models"]/f"{name}_fold{fold}.pt")
        vi,pv,zv=graph_predict(model,PyGDataLoader([MOLECULAR_GRAPHS[i] for i in vag],batch_size=CFG["batch_size"],shuffle=False))
        ti,pt,zt=graph_predict(model,PyGDataLoader([MOLECULAR_GRAPHS[i] for i in test_idx],batch_size=CFG["batch_size"],shuffle=False))
        map_v={g:i for i,g in enumerate(vag)}; order_v=np.argsort([map_v[int(i)] for i in vi]); oof[val]=pv[order_v]; oof_z[val]=zv[order_v]
        map_t={g:i for i,g in enumerate(test_idx)}; order_t=np.argsort([map_t[int(i)] for i in ti]); test_ps.append(pt[order_t]);test_zs.append(zt[order_t])
        print(f"{name}: fold {fold+1}, best AUC={best:.4f}")
        del model; gc.collect();
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    tp=np.mean(test_ps,axis=0); tz=np.mean(test_zs,axis=0)
    save_model_artifact(name,oof,tp,oof_z,tz,{"type":"molecular_graph","cv":N_CV_FOLDS})
    return oof,tp,oof_z,tz

In [ ]:
if RUN_MOLECULAR_GNN and PYG_AVAILABLE:
    ATTFP_OOF,ATTFP_TEST,ATTFP_OOF_EMB,ATTFP_TEST_EMB=run_graph_branch_cv(
        "AttentiveFP",lambda:AttentiveFPModel(len(TARGET_COLS),hidden=192 if LOW_MEMORY else 256,layers=4),"focal")
    MPNN_OOF,MPNN_TEST,MPNN_OOF_EMB,MPNN_TEST_EMB=run_graph_branch_cv(
        "MPNN",lambda:MPNNModel(len(TARGET_COLS),hidden=192 if LOW_MEMORY else 256,steps=4),"asymmetric")
elif RUN_MOLECULAR_GNN:
    print("PyG unavailable হওয়ায় molecular GNN branch skip হয়েছে। Kernel restart করে পুনরায় চালান।")

## 21. ChemBERTa multitask branch

The notebook downloads a public pretrained ChemBERTa checkpoint when first used. It starts with a frozen encoder, trains the task head, then gradually unfreezes top transformer layers. Randomized SMILES are used only during training; canonical SMILES are used for validation/test.

In [ ]:
try:
    from transformers import AutoTokenizer, AutoModel, AutoModelForMaskedLM, DataCollatorForLanguageModeling, get_cosine_schedule_with_warmup
    TRANSFORMERS_AVAILABLE=True
except Exception as e:
    TRANSFORMERS_AVAILABLE=False; print(e)

CHEMBERTA_CHECKPOINT="DeepChem/ChemBERTa-77M-MLM"

class _MLMSmilesDataset(Dataset):
    def __init__(self,smiles,tokenizer,max_len=256):self.smiles=list(smiles);self.tok=tokenizer;self.max_len=max_len
    def __len__(self):return len(self.smiles)
    def __getitem__(self,i):return self.tok(self.smiles[i],truncation=True,max_length=self.max_len,return_special_tokens_mask=True)

def fold_local_domain_adapt(base_checkpoint,train_smiles,fold,seed=13):
    """Strict fold-local unsupervised MLM adaptation; no validation/test SMILES are used."""
    out=DIRS["models"]/f"ChemBERTa_MLM_fold{fold}"
    if out.exists():return str(out)
    if not RUN_CHEMBERTA_DOMAIN_ADAPTATION:return base_checkpoint
    print(f"বাংলা বার্তা: Fold {fold}-এর training SMILES দিয়ে ChemBERTa domain-adaptive MLM pretraining হচ্ছে।")
    tok=AutoTokenizer.from_pretrained(base_checkpoint);model=AutoModelForMaskedLM.from_pretrained(base_checkpoint).to(DEVICE)
    smiles=list(train_smiles)
    if EXECUTION_MODE=="QUICK":smiles=smiles[:min(1000,len(smiles))]
    ds=_MLMSmilesDataset(smiles,tok,192 if LOW_MEMORY else 256);coll=DataCollatorForLanguageModeling(tok,mlm=True,mlm_probability=.15)
    loader=DataLoader(ds,batch_size=8 if LOW_MEMORY else 32,shuffle=True,collate_fn=coll)
    opt=torch.optim.AdamW(model.parameters(),lr=2e-5,weight_decay=1e-5);epochs=1 if EXECUTION_MODE=="QUICK" else (2 if EXECUTION_MODE=="STANDARD" else 3)
    model.train()
    for _ in range(epochs):
        for b in loader:
            b={k:v.to(DEVICE) for k,v in b.items()};opt.zero_grad(set_to_none=True);loss=model(**b).loss;loss.backward();nn.utils.clip_grad_norm_(model.parameters(),1);opt.step()
    model.save_pretrained(out);tok.save_pretrained(out);del model;gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()
    return str(out)

class SmilesDataset(Dataset):
    def __init__(self,smiles,y,mask,indices,tokenizer,max_len=256,augment=False,seed=13):
        self.smiles=list(smiles);self.y=y;self.mask=mask;self.indices=np.asarray(indices);self.tok=tokenizer;self.max_len=max_len;self.augment=augment;self.seed=seed
    def __len__(self):return len(self.smiles)
    def __getitem__(self,i):
        s=self.smiles[i]
        if self.augment: s=randomized_smiles(s,1,self.seed+i)[0]
        enc=self.tok(s,truncation=True,padding="max_length",max_length=self.max_len,return_tensors="pt")
        return {"input_ids":enc["input_ids"].squeeze(0),"attention_mask":enc["attention_mask"].squeeze(0),
                "y":torch.tensor(self.y[i]),"mask":torch.tensor(self.mask[i]),"idx":torch.tensor(self.indices[i])}

class ChemBERTaMultitask(nn.Module):
    def __init__(self,checkpoint,n_tasks,embed_dim=256,dropout=0.2):
        super().__init__(); self.encoder=AutoModel.from_pretrained(checkpoint)
        h=self.encoder.config.hidden_size; self.proj=nn.Sequential(nn.Linear(h,embed_dim),nn.LayerNorm(embed_dim),nn.GELU(),nn.Dropout(dropout)); self.head=nn.Linear(embed_dim,n_tasks)
    def freeze_encoder(self):
        for p in self.encoder.parameters(): p.requires_grad=False
    def unfreeze_top(self,n_layers=2):
        for p in self.encoder.parameters(): p.requires_grad=False
        layers=getattr(getattr(self.encoder,"encoder",None),"layer",None)
        if layers is not None:
            for layer in layers[-n_layers:]:
                for p in layer.parameters(): p.requires_grad=True
        if hasattr(self.encoder,"pooler") and self.encoder.pooler:
            for p in self.encoder.pooler.parameters(): p.requires_grad=True
    def forward(self,input_ids,attention_mask):
        o=self.encoder(input_ids=input_ids,attention_mask=attention_mask)
        h=o.last_hidden_state; m=attention_mask.unsqueeze(-1).float(); pooled=(h*m).sum(1)/m.sum(1).clamp_min(1)
        z=self.proj(pooled); return self.head(z),z

In [ ]:
def predict_chemberta(model,loader):
    model.eval(); rec=[]
    with torch.no_grad():
        for b in loader:
            ids=b["input_ids"].to(DEVICE);am=b["attention_mask"].to(DEVICE)
            with torch.amp.autocast("cuda",enabled=USE_AMP): logits,z=model(ids,am)
            for idx,p,e in zip(b["idx"].numpy(),torch.sigmoid(logits).cpu().numpy(),z.cpu().numpy()): rec.append((int(idx),p,e))
    rec.sort(key=lambda x:x[0]); return np.array([r[0] for r in rec]),np.stack([r[1] for r in rec]),np.stack([r[2] for r in rec])

def train_chemberta_fold(tr_global,va_global,fold,seed=13):
    print("ChemBERTa pretrained model/tokenizer প্রথমবার download হতে পারে...")
    fold_checkpoint=fold_local_domain_adapt(CHEMBERTA_CHECKPOINT,curated_df.loc[tr_global,"canonical_smiles"],fold,seed)
    tok=AutoTokenizer.from_pretrained(fold_checkpoint)
    model=ChemBERTaMultitask(fold_checkpoint,len(TARGET_COLS),256).to(DEVICE)
    model.freeze_encoder()
    max_len=192 if LOW_MEMORY else 256; bs=4 if LOW_MEMORY else 16
    trds=SmilesDataset(curated_df.loc[tr_global,"canonical_smiles"],Y_FILLED[tr_global],M[tr_global],tr_global,tok,max_len,True,seed)
    vads=SmilesDataset(curated_df.loc[va_global,"canonical_smiles"],Y_FILLED[va_global],M[va_global],va_global,tok,max_len,False,seed)
    trl=DataLoader(trds,batch_size=bs,shuffle=True); val=DataLoader(vads,batch_size=bs*2,shuffle=False)
    wpos,wneg=effective_num_weights(torch.tensor(Y_FILLED[tr_global],device=DEVICE),torch.tensor(M[tr_global],device=DEVICE)); criterion=MaskedClassBalancedFocalLoss(wpos,wneg)

    def phase(epochs,lr,unfreeze=False):
        if unfreeze: model.unfreeze_top(2 if LOW_MEMORY else 3)
        params=[p for p in model.parameters() if p.requires_grad];opt=torch.optim.AdamW(params,lr=lr,weight_decay=1e-5)
        total=max(1,epochs*len(trl));sched=get_cosine_schedule_with_warmup(opt,int(.08*total),total)
        scaler=torch.amp.GradScaler("cuda",enabled=USE_AMP);best=-np.inf;state=None;wait=0
        for ep in range(epochs):
            model.train()
            for b in trl:
                ids=b["input_ids"].to(DEVICE);am=b["attention_mask"].to(DEVICE);y=b["y"].to(DEVICE);mm=b["mask"].to(DEVICE)
                opt.zero_grad(set_to_none=True)
                with torch.amp.autocast("cuda",enabled=USE_AMP): lg,_=model(ids,am);loss=criterion(lg,y,mm)
                scaler.scale(loss).backward();scaler.unscale_(opt);nn.utils.clip_grad_norm_(model.parameters(),1);scaler.step(opt);scaler.update();sched.step()
            _,pv,_=predict_chemberta(model,val);auc=mean_masked_auc(Y_FILLED[va_global],M[va_global],pv)
            if auc>best+1e-5: best=auc;state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()};wait=0
            else:wait+=1
            if wait>=max(2,CFG["patience"]//2):break
        if state:model.load_state_dict(state)
        return best
    head_epochs=2 if EXECUTION_MODE=="QUICK" else min(10,CFG["epochs"]//4)
    b1=phase(head_epochs,3e-4,False)
    b2=phase(max(2,CFG["epochs"]-head_epochs),2e-5,True)
    return model,tok,max(b1,b2)

In [ ]:
def run_chemberta_cv():
    oof=np.full((len(train_df),len(TARGET_COLS)),np.nan,np.float32);oofz=np.zeros((len(train_df),256),np.float32);tps=[];tzs=[]
    for fold in range(N_CV_FOLDS):
        trl=np.where(train_df.cv_fold.values!=fold)[0];val=np.where(train_df.cv_fold.values==fold)[0]
        trg=train_df.global_index.values[trl];vag=train_df.global_index.values[val]
        model,tok,best=train_chemberta_fold(trg,vag,fold,PRIMARY_SEED+fold)
        model.save_pretrained(DIRS["models"]/f"ChemBERTa_fold{fold}")
        tok.save_pretrained(DIRS["models"]/f"ChemBERTa_fold{fold}")
        max_len=192 if LOW_MEMORY else 256;bs=8 if LOW_MEMORY else 32
        va_loader=DataLoader(SmilesDataset(curated_df.loc[vag,"canonical_smiles"],Y_FILLED[vag],M[vag],vag,tok,max_len,False),batch_size=bs)
        te_loader=DataLoader(SmilesDataset(curated_df.loc[test_idx,"canonical_smiles"],Y_FILLED[test_idx],M[test_idx],test_idx,tok,max_len,False),batch_size=bs)
        vi,pv,zv=predict_chemberta(model,va_loader);ti,pt,zt=predict_chemberta(model,te_loader)
        vm={g:i for i,g in enumerate(vag)};order=np.argsort([vm[i] for i in vi]);oof[val]=pv[order];oofz[val]=zv[order]
        tm={g:i for i,g in enumerate(test_idx)};order=np.argsort([tm[i] for i in ti]);tps.append(pt[order]);tzs.append(zt[order])
        print(f"ChemBERTa fold {fold+1}: best AUC={best:.4f}")
        del model;gc.collect();
        if torch.cuda.is_available():torch.cuda.empty_cache()
    tp=np.mean(tps,axis=0);tz=np.mean(tzs,axis=0)
    save_model_artifact("ChemBERTa",oof,tp,oofz,tz,{"checkpoint":CHEMBERTA_CHECKPOINT,"cv":N_CV_FOLDS,"staged_unfreezing":True})
    return oof,tp,oofz,tz

if RUN_CHEMBERTA and TRANSFORMERS_AVAILABLE:
    print("বাংলা বার্তা: ChemBERTa pretrained weights প্রথমবার Hugging Face থেকে download হবে; internet প্রয়োজন।")
    CHEM_OOF,CHEM_TEST,CHEM_OOF_EMB,CHEM_TEST_EMB=run_chemberta_cv()
elif RUN_CHEMBERTA:
    print("Transformers package unavailable; ChemBERTa skip হয়েছে।")

## 22. Optional DenseNet121 2D molecular-image branch

Standardized RDKit depictions are generated on the fly at 224×224. ImageNet-pretrained DenseNet121 is first trained with a frozen backbone, then its final dense block is partially unfrozen. The branch is retained only if it adds OOF value or ensemble diversity. Grad-CAM hooks are included for explanation.

In [ ]:
try:
    import torchvision
    from torchvision.models import densenet121, DenseNet121_Weights
    from torchvision import transforms
    from PIL import Image
    TORCHVISION_AVAILABLE=True
except Exception as e:
    TORCHVISION_AVAILABLE=False
    print("torchvision unavailable:",e)

class MoleculeImageDataset(Dataset):
    def __init__(self,global_indices,augment=False):
        self.ids=np.asarray(global_indices);self.augment=augment
        self.tf=transforms.Compose([
            transforms.Resize((224,224)),
            transforms.RandomRotation(8) if augment else transforms.Lambda(lambda x:x),
            transforms.ToTensor(),
            transforms.Normalize([.5,.5,.5],[.5,.5,.5])
        ]) if TORCHVISION_AVAILABLE else None
    def __len__(self):return len(self.ids)
    def __getitem__(self,i):
        gi=int(self.ids[i]);mol=Chem.MolFromSmiles(curated_df.loc[gi,"canonical_smiles"])
        img=Draw.MolToImage(mol,size=(224,224)).convert("RGB")
        return self.tf(img),torch.tensor(Y_FILLED[gi]),torch.tensor(M[gi]),torch.tensor(gi)

class DenseNet121Multitask(nn.Module):
    def __init__(self,n_tasks,embed_dim=256,pretrained=True):
        super().__init__()
        weights=DenseNet121_Weights.IMAGENET1K_V1 if pretrained else None
        self.backbone=densenet121(weights=weights)
        d=self.backbone.classifier.in_features
        self.backbone.classifier=nn.Identity()
        self.embed=nn.Sequential(nn.Linear(d,embed_dim),nn.LayerNorm(embed_dim),nn.GELU(),nn.Dropout(.25))
        self.head=nn.Linear(embed_dim,n_tasks)
    def freeze_backbone(self):
        for p in self.backbone.parameters():p.requires_grad=False
    def unfreeze_last_block(self):
        for p in self.backbone.parameters():p.requires_grad=False
        for p in self.backbone.features.denseblock4.parameters():p.requires_grad=True
        for p in self.backbone.features.norm5.parameters():p.requires_grad=True
    def forward(self,x):
        z=self.embed(self.backbone(x));return self.head(z),z

In [ ]:
def train_image_fold(trg,vag,seed=13):
    print("বাংলা বার্তা: DenseNet121 pretrained ImageNet weights প্রথমবার download হতে পারে।")
    model=DenseNet121Multitask(len(TARGET_COLS),256,pretrained=True).to(DEVICE);model.freeze_backbone()
    trl=DataLoader(MoleculeImageDataset(trg,True),batch_size=8 if LOW_MEMORY else 24,shuffle=True,num_workers=0)
    val=DataLoader(MoleculeImageDataset(vag,False),batch_size=16 if LOW_MEMORY else 48,shuffle=False,num_workers=0)
    wpos,wneg=effective_num_weights(torch.tensor(Y_FILLED[trg],device=DEVICE),torch.tensor(M[trg],device=DEVICE));crit=MaskedClassBalancedFocalLoss(wpos,wneg)
    def phase(epochs,lr,unfreeze=False):
        if unfreeze:model.unfreeze_last_block()
        opt=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=lr,weight_decay=1e-5)
        best=-np.inf;state=None;wait=0;scaler=torch.amp.GradScaler("cuda",enabled=USE_AMP)
        for ep in range(epochs):
            model.train()
            for x,y,m,_ in trl:
                x,y,m=x.to(DEVICE),y.to(DEVICE),m.to(DEVICE);opt.zero_grad(set_to_none=True)
                with torch.amp.autocast("cuda",enabled=USE_AMP):lg,_=model(x);loss=crit(lg,y,m)
                scaler.scale(loss).backward();scaler.unscale_(opt);nn.utils.clip_grad_norm_(model.parameters(),1);scaler.step(opt);scaler.update()
            ids,p,_=predict_image_model(model,val);order=np.argsort(ids);p=p[order];auc=mean_masked_auc(Y_FILLED[np.sort(vag)],M[np.sort(vag)],p)
            if auc>best+1e-5:best=auc;state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()};wait=0
            else:wait+=1
            if wait>=max(2,CFG["patience"]//2):break
        if state:model.load_state_dict(state)
        return best
    b1=phase(2 if EXECUTION_MODE=="QUICK" else 8,3e-4,False)
    b2=phase(2 if EXECUTION_MODE=="QUICK" else max(8,CFG["epochs"]//3),2e-5,True)
    return model,max(b1,b2)

def predict_image_model(model,loader):
    model.eval();rec=[]
    with torch.no_grad():
        for x,_,_,ids in loader:
            x=x.to(DEVICE)
            with torch.amp.autocast("cuda",enabled=USE_AMP):lg,z=model(x)
            for i,p,e in zip(ids.numpy(),torch.sigmoid(lg).cpu().numpy(),z.cpu().numpy()):rec.append((int(i),p,e))
    rec.sort(key=lambda x:x[0]);return np.array([r[0] for r in rec]),np.stack([r[1] for r in rec]),np.stack([r[2] for r in rec])

def run_densenet_cv():
    oof=np.full((len(train_df),len(TARGET_COLS)),np.nan,np.float32);oofz=np.zeros((len(train_df),256),np.float32);tps=[];tzs=[]
    for fold in range(N_CV_FOLDS):
        trl=np.where(train_df.cv_fold.values!=fold)[0];val=np.where(train_df.cv_fold.values==fold)[0];trg=train_df.global_index.values[trl];vag=train_df.global_index.values[val]
        model,best=train_image_fold(trg,vag,PRIMARY_SEED+fold)
        torch.save({"state_dict":model.state_dict(),"fold":fold},DIRS["models"]/f"DenseNet121_fold{fold}.pt")
        vi,pv,zv=predict_image_model(model,DataLoader(MoleculeImageDataset(vag,False),batch_size=32));ti,pt,zt=predict_image_model(model,DataLoader(MoleculeImageDataset(test_idx,False),batch_size=32))
        vm={g:i for i,g in enumerate(vag)};ov=np.argsort([vm[i] for i in vi]);oof[val]=pv[ov];oofz[val]=zv[ov]
        tm={g:i for i,g in enumerate(test_idx)};ot=np.argsort([tm[i] for i in ti]);tps.append(pt[ot]);tzs.append(zt[ot]);print(f"DenseNet121 fold {fold+1}: {best:.4f}")
        del model;gc.collect();
        if torch.cuda.is_available():torch.cuda.empty_cache()
    tp=np.mean(tps,0);tz=np.mean(tzs,0);save_model_artifact("DenseNet121",oof,tp,oofz,tz,{"type":"2D_image","cv":N_CV_FOLDS});return oof,tp,oofz,tz

if RUN_OPTIONAL_DENSENET and TORCHVISION_AVAILABLE:
    DENSENET_RESULT=run_densenet_cv()
elif RUN_OPTIONAL_DENSENET:
    print("DenseNet121 skipped because torchvision is unavailable.")

## 23. Optional 3D EGNN branch

This branch generates ETKDG conformers, optimizes geometry and applies a lightweight E(n)-equivariant message-passing network. It is disabled by default until the core pipeline is stable; enable it only if OOF gain justifies the compute.

In [ ]:
def smiles_to_3d_graph(smiles,y_row,mask_row,idx,seed=13):
    mol=Chem.AddHs(Chem.MolFromSmiles(smiles));params=AllChem.ETKDGv3();params.randomSeed=int(seed)
    ok=AllChem.EmbedMolecule(mol,params)
    if ok!=0:return None
    try:AllChem.MMFFOptimizeMolecule(mol,maxIters=200)
    except:pass
    conf=mol.GetConformer();pos=np.array([list(conf.GetAtomPosition(i)) for i in range(mol.GetNumAtoms())],np.float32)
    z=np.array([a.GetAtomicNum() for a in mol.GetAtoms()],np.int64)
    src=[];dst=[]
    dist=np.linalg.norm(pos[:,None,:]-pos[None,:,:],axis=-1)
    for i in range(len(z)):
        for j in range(len(z)):
            if i!=j and dist[i,j] <= 5.0:src.append(i);dst.append(j)
    d=Data(z=torch.tensor(z),pos=torch.tensor(pos),edge_index=torch.tensor([src,dst]),y=torch.tensor(y_row).view(1,-1),mask=torch.tensor(mask_row).view(1,-1),sample_idx=torch.tensor([idx]))
    return d

class EGNNLayer(nn.Module):
    def __init__(self,h):
        super().__init__();self.edge=nn.Sequential(nn.Linear(h*2+1,h),nn.SiLU(),nn.Linear(h,h));self.coord=nn.Sequential(nn.Linear(h,h),nn.SiLU(),nn.Linear(h,1));self.node=nn.Sequential(nn.Linear(h*2,h),nn.SiLU(),nn.Linear(h,h))
    def forward(self,h,pos,ei):
        s,t=ei;diff=pos[s]-pos[t];d2=(diff**2).sum(-1,keepdim=True);m=self.edge(torch.cat([h[s],h[t],d2],-1));coef=self.coord(m)
        agg=torch.zeros_like(h).index_add_(0,t,m);delta=torch.zeros_like(pos).index_add_(0,t,diff*coef)
        return h+self.node(torch.cat([h,agg],-1)),pos+delta

class EGNNModel(nn.Module):
    def __init__(self,n_tasks,h=192,layers=4):
        super().__init__();self.emb=nn.Embedding(120,h);self.layers=nn.ModuleList([EGNNLayer(h) for _ in range(layers)]);self.proj=nn.Linear(h,256);self.head=nn.Linear(256,n_tasks)
    def forward(self,d):
        h=self.emb(d.z.clamp_max(119));pos=d.pos
        for layer in self.layers:h,pos=layer(h,pos,d.edge_index)
        g=global_mean_pool(h,d.batch);z=torch.relu(self.proj(g));return self.head(z),z

def run_egnn_cv():
    cache=DIRS["cache"]/"egnn_3d_graphs_v1.pt"
    if cache.exists():graphs=torch.load(cache,weights_only=False)
    else:
        print("বাংলা বার্তা: 3D ETKDG conformer generate করা হচ্ছে; এটি সময়সাপেক্ষ।")
        graphs={}
        for i,smi in enumerate(tqdm(curated_df.canonical_smiles,desc="3D conformers")):
            g=smiles_to_3d_graph(smi,Y_FILLED[i],M[i],i,PRIMARY_SEED)
            if g is not None:graphs[i]=g
        torch.save(graphs,cache)
    valid_train=np.array([i for i,g in enumerate(train_df.global_index.values) if g in graphs]);valid_test=np.array([g for g in test_idx if g in graphs]);test_pos={g:i for i,g in enumerate(test_idx)}
    oof=np.full((len(train_df),len(TARGET_COLS)),np.nan,np.float32);oofz=np.zeros((len(train_df),256),np.float32);tps=[];tzs=[]
    for fold in range(N_CV_FOLDS):
        trl=np.array([i for i in valid_train if train_df.cv_fold.iloc[i]!=fold]);val=np.array([i for i in valid_train if train_df.cv_fold.iloc[i]==fold]);trg=train_df.global_index.values[trl];vag=train_df.global_index.values[val]
        model,best=train_graph_model(EGNNModel(len(TARGET_COLS),h=128 if LOW_MEMORY else 192,layers=3 if LOW_MEMORY else 4),[graphs[i] for i in trg],[graphs[i] for i in vag],PRIMARY_SEED+fold,"focal")
        torch.save({"state_dict":model.state_dict(),"fold":fold},DIRS["models"]/f"EGNN_fold{fold}.pt")
        vi,pv,zv=graph_predict(model,PyGDataLoader([graphs[i] for i in vag],batch_size=CFG["batch_size"]));ti,pt,zt=graph_predict(model,PyGDataLoader([graphs[i] for i in valid_test],batch_size=CFG["batch_size"]))
        vm={g:i for i,g in enumerate(vag)};ov=np.argsort([vm[int(i)] for i in vi]);oof[val]=pv[ov];oofz[val]=zv[ov]
        ft=np.full((len(test_idx),len(TARGET_COLS)),np.nan,np.float32);fz=np.zeros((len(test_idx),256),np.float32)
        for g,p,e in zip(ti,pt,zt):ft[test_pos[int(g)]]=p;fz[test_pos[int(g)]]=e
        tps.append(ft);tzs.append(fz);print(f"EGNN fold {fold+1}: {best:.4f}")
    tp=np.nanmean(np.stack(tps),0);tz=np.mean(tzs,0);save_model_artifact("EGNN_3D",oof,tp,oofz,tz,{"type":"3D_equivariant","cv":N_CV_FOLDS});return oof,tp,oofz,tz

if RUN_OPTIONAL_3D_EGNN and PYG_AVAILABLE:
    EGNN_RESULT=run_egnn_cv()
else:
    print("3D EGNN code ready; branch is disabled until core-model OOF gate is passed.")

# Part IV — Leakage-safe toxicological knowledge graph

The notebook downloads the paper authors' public repository when enabled. It searches for `common_cids.csv`, `compound_master.csv`, `filtered_triples.csv`, gene/pathway tables and official model code. Target Tox21 active/inactive assay outcome edges and label-derived features are removed before graph construction.

## 22. Download the public ToxKG repository snapshot

The public repository is large. The first run can take time. All downloads are cached under `knowledge_graph/raw/`.

In [ ]:
TOXKG_REPO_ID="xiejunjie/Molecular-toxicity-prediction"
TOXKG_SNAPSHOT=DIRS["kg_raw"] / "Molecular-toxicity-prediction"

if RUN_KG_MODELS and AUTO_DOWNLOAD_TOXKG and not TOXKG_SNAPSHOT.exists():
    print("বাংলা বার্তা: ToxKG paper-এর public Hugging Face repository download করা হচ্ছে। এটি বড় হতে পারে এবং internet প্রয়োজন।")
    try:
        from huggingface_hub import snapshot_download
        snapshot_download(repo_id=TOXKG_REPO_ID,repo_type="model",local_dir=str(TOXKG_SNAPSHOT),local_dir_use_symlinks=False)
        print("ToxKG repository download সম্পন্ন হয়েছে।")
    except Exception as e:
        print("ToxKG repository download ব্যর্থ হয়েছে:",e)
        print("Manual option: Hugging Face থেকে repository download করে knowledge_graph/raw/Molecular-toxicity-prediction ফোল্ডারে রাখুন।")
elif RUN_KG_MODELS:
    print("ToxKG snapshot path:",TOXKG_SNAPSHOT)

In [ ]:
def locate_repo_file(patterns):
    if not TOXKG_SNAPSHOT.exists():return None
    for pat in patterns:
        hits=list(TOXKG_SNAPSHOT.rglob(pat))
        if hits:return hits[0]
    return None

KG_FILES={
    "triples":locate_repo_file(["filtered_triples.csv","*triples*.csv"]),
    "compound":locate_repo_file(["compound_master.csv","common_cids.csv"]),
    "fingerprints":locate_repo_file(["gnn_input_fp.csv"]),
    "genes":locate_repo_file(["gnn_input_genes.csv","*genes*.csv"]),
    "pathways":locate_repo_file(["gnn_input_pathways.csv","*pathways*.csv"]),
}
print("Detected KG files:")
for k,v in KG_FILES.items():print(f"  {k}: {v}")

## 23. Mandatory KG leakage audit

Pass conditions:

- No relation directly encodes any of the 12 target outcomes.
- No `CHEMICALHASACTIVEASSAY` / `CHEMICALHASINACTIVEASSAY` target edge remains.
- No node feature is generated from the target labels.
- No molecule/InChIKey/scaffold overlap crosses the frozen split.
- A label-permutation model returns approximately chance AUC.

In [ ]:
TARGET_TOKENS={re.sub(r"[^a-z0-9]","",t.lower()) for t in TARGET_COLS}
SUSPECT_RELATION_PATTERNS=["activeassay","inactiveassay","hasactive","hasinactive","tox21label","targetlabel","activitylabel"]

def normalize_token(x):return re.sub(r"[^a-z0-9]","",str(x).lower())

def audit_and_filter_triples(triples):
    cols={c.lower():c for c in triples.columns}
    def find_col(keys):
        for k in keys:
            for c in triples.columns:
                if k in c.lower():return c
        return None
    src=find_col(["source","head","start","subject","from"])
    rel=find_col(["relation","predicate","edge","type"])
    dst=find_col(["target","tail","end","object","to"])
    if not all([src,rel,dst]):
        raise ValueError(f"Triple columns detect করা যায়নি: {triples.columns.tolist()}")
    rel_norm=triples[rel].astype(str).map(normalize_token)
    suspect=np.zeros(len(triples),dtype=bool)
    for p in SUSPECT_RELATION_PATTERNS:suspect|=rel_norm.str.contains(p,regex=False).values
    # Also reject relations that combine a target endpoint token with assay/activity/outcome semantics.
    for tok in TARGET_TOKENS:
        suspect |= (rel_norm.str.contains(tok,regex=False) & rel_norm.str.contains("assay|active|inactive|outcome",regex=True)).values
    safe=triples.loc[~suspect].copy();removed=triples.loc[suspect].copy()
    return safe,removed,{"source":src,"relation":rel,"target":dst}

SAFE_TRIPLES=None;TRIPLE_COLS=None
if KG_FILES["triples"]:
    triples_raw=pd.read_csv(KG_FILES["triples"])
    SAFE_TRIPLES,REMOVED_TRIPLES,TRIPLE_COLS=audit_and_filter_triples(triples_raw)
    SAFE_TRIPLES.to_csv(DIRS["kg_processed"] / "safe_triples.csv",index=False)
    REMOVED_TRIPLES.to_csv(DIRS["kg_provenance"] / "removed_target_or_suspect_edges.csv",index=False)
    print(f"KG triples: raw={len(triples_raw):,}, safe={len(SAFE_TRIPLES):,}, removed={len(REMOVED_TRIPLES):,}")
    if len(REMOVED_TRIPLES):display(REMOVED_TRIPLES.head())
else:
    print("KG triple file পাওয়া যায়নি; KG branch পরবর্তী cell-এ graceful skip করবে।")

In [ ]:
# Automated leakage audit report
leakage_report={
    "molecule_overlap":len(train_ik & test_ik),
    "scaffold_overlap":len(train_scaffolds & test_scaffolds),
    "target_edges_removed":int(len(REMOVED_TRIPLES)) if "REMOVED_TRIPLES" in globals() else None,
    "safe_triples_available":SAFE_TRIPLES is not None,
    "full_dataset_feature_fit":False,
    "test_used_for_tuning":False,
}
leakage_report["pass"]=(leakage_report["molecule_overlap"]==0 and leakage_report["scaffold_overlap"]==0)
with open(DIRS["outputs"] / "leakage_audit.json","w") as f:json.dump(leakage_report,f,indent=2)
display(pd.DataFrame([leakage_report]))
assert leakage_report["pass"],"Leakage audit failed"

## 24. KG cohort mapping and safe mechanism matrix

This cell maps the paper repository compounds to the curated Tox21 identities. It creates a sparse chemical–gene/pathway mechanism matrix from safe triples. This matrix is a reproducible fallback and also supplies node features for KG models.

In [ ]:
def detect_col(df,candidates):
    for c in df.columns:
        if c.lower() in [x.lower() for x in candidates]:return c
    for c in df.columns:
        if any(x.lower() in c.lower() for x in candidates):return c
    return None

KG_COMPOUND_DF=None
if KG_FILES["compound"]:
    KG_COMPOUND_DF=pd.read_csv(KG_FILES["compound"])
    kg_smiles_col=detect_col(KG_COMPOUND_DF,["smiles","canonical_smiles"])
    kg_cid_col=detect_col(KG_COMPOUND_DF,["cid","pubchem_cid","compound_id"])
    if kg_smiles_col:
        print("KG compound SMILES standardize করা হচ্ছে...")
        KG_COMPOUND_DF["canonical_smiles"]=[standardize_smiles(s)["canonical_smiles"] for s in tqdm(KG_COMPOUND_DF[kg_smiles_col],desc="KG SMILES")]
    display(KG_COMPOUND_DF.head(2))
else:
    kg_smiles_col=kg_cid_col=None

In [ ]:
KG_MATCHED_GLOBAL=np.array([],dtype=int)
KG_COVERAGE=np.zeros(len(curated_df),dtype=np.float32)
KG_MECHANISM=None

if SAFE_TRIPLES is not None and KG_COMPOUND_DF is not None and kg_smiles_col:
    # Map curated molecules to KG chemical identifiers.
    cur_map={s:i for i,s in enumerate(curated_df["canonical_smiles"])}
    kg_rows=KG_COMPOUND_DF[KG_COMPOUND_DF["canonical_smiles"].isin(cur_map)].copy()
    KG_MATCHED_GLOBAL=np.array([cur_map[s] for s in kg_rows["canonical_smiles"]],dtype=int)
    KG_COVERAGE[KG_MATCHED_GLOBAL]=1.0
    print(f"KG-matched curated molecules: {len(KG_MATCHED_GLOBAL):,}/{len(curated_df):,}")

    src,rel,dst=TRIPLE_COLS["source"],TRIPLE_COLS["relation"],TRIPLE_COLS["target"]
    chemical_ids=set()
    if kg_cid_col: chemical_ids=set(KG_COMPOUND_DF[kg_cid_col].astype(str))
    # Create relation-aware neighbor tokens: relation::neighbor, preserving semantics without target outcomes.
    row_map={str(v):cur_map[s] for v,s in zip(kg_rows[kg_cid_col].astype(str),kg_rows["canonical_smiles"])} if kg_cid_col else {}
    features=defaultdict(list)
    for _,r in SAFE_TRIPLES[[src,rel,dst]].iterrows():
        a,b,rr=str(r[src]),str(r[dst]),str(r[rel])
        if a in row_map:features[row_map[a]].append(f"OUT::{rr}::{b}")
        if b in row_map:features[row_map[b]].append(f"IN::{rr}::{a}")
    vocab={tok:j for j,tok in enumerate(sorted({x for vals in features.values() for x in vals}))}
    rows=[];cols=[];vals=[]
    for gi,toks in features.items():
        for tok,count in Counter(toks).items():rows.append(gi);cols.append(vocab[tok]);vals.append(count)
    KG_MECHANISM=sparse.csr_matrix((vals,(rows,cols)),shape=(len(curated_df),len(vocab)),dtype=np.float32)
    sparse.save_npz(DIRS["kg_processed"] / "safe_mechanism_matrix.npz",KG_MECHANISM)
    pd.DataFrame({"token":list(vocab.keys()),"column":list(vocab.values())}).to_csv(DIRS["kg_processed"] / "mechanism_vocab.csv",index=False)
    print("Safe relation-aware mechanism matrix:",KG_MECHANISM.shape, "nnz=",KG_MECHANISM.nnz)
else:
    print("KG inputs incomplete; safe KG mechanism matrix তৈরি হয়নি। Non-KG branches will still run.")

## 25. KG branch models: R-GCN, HGT and GPS

The code below provides three relation-aware KG encoders. For scalable execution, each chemical is represented by a leakage-safe local mechanism graph derived from safe triples. If the public KG files are unavailable, the section skips without breaking the rest of the notebook.

In [ ]:
# Local mechanism graph builder from safe relation-aware tokens.
# It is deliberately target-edge free. Chemical root is node 0; neighbor nodes are type/relation aware.
KG_LOCAL_GRAPHS=None
KG_RELATION_VOCAB={}
KG_NODE_TOKEN_VOCAB={}

if PYG_AVAILABLE and SAFE_TRIPLES is not None and KG_COMPOUND_DF is not None and kg_cid_col and kg_smiles_col:
    src,rel,dst=TRIPLE_COLS["source"],TRIPLE_COLS["relation"],TRIPLE_COLS["target"]
    cur_map={s:i for i,s in enumerate(curated_df["canonical_smiles"])}
    kg_rows=KG_COMPOUND_DF[KG_COMPOUND_DF["canonical_smiles"].isin(cur_map)].copy()
    id_to_global={str(v):cur_map[s] for v,s in zip(kg_rows[kg_cid_col].astype(str),kg_rows["canonical_smiles"])}
    adj=defaultdict(list)
    for _,r in SAFE_TRIPLES[[src,rel,dst]].iterrows():
        a,b,rr=str(r[src]),str(r[dst]),str(r[rel])
        if a in id_to_global:adj[id_to_global[a]].append((rr,b,"out"))
        if b in id_to_global:adj[id_to_global[b]].append((rr,a,"in"))
    KG_RELATION_VOCAB={r:i for i,r in enumerate(sorted({x[0] for vals in adj.values() for x in vals}))}
    all_tokens=sorted({x[1] for vals in adj.values() for x in vals})
    KG_NODE_TOKEN_VOCAB={t:i+1 for i,t in enumerate(all_tokens)}
    def make_kg_local_graph(global_idx,max_neighbors=128):
        neigh=adj.get(global_idx,[])[:max_neighbors]
        # node 0 = chemical; neighbors have hashed global token IDs.
        node_ids=[0]+[KG_NODE_TOKEN_VOCAB[n] for _,n,_ in neigh]
        node_types=[0]+[1 if "gene" in n.lower() else 2 if "path" in n.lower() else 3 for _,n,_ in neigh]
        edges=[];et=[]
        for k,(r,n,direction) in enumerate(neigh,1):
            rid=KG_RELATION_VOCAB[r]
            if direction=="out":edges.append([0,k]);et.append(rid)
            else:edges.append([k,0]);et.append(rid)
        if not edges:edges=[[0,0]];et=[0]
        data=Data(node_id=torch.tensor(node_ids),node_type=torch.tensor(node_types),edge_index=torch.tensor(edges).t().contiguous(),edge_type=torch.tensor(et))
        data.y=torch.tensor(Y_FILLED[global_idx]).view(1,-1);data.mask=torch.tensor(M[global_idx]).view(1,-1);data.sample_idx=torch.tensor([global_idx]);data.root_index=torch.tensor([0])
        return data
    KG_LOCAL_GRAPHS={i:make_kg_local_graph(i) for i in KG_MATCHED_GLOBAL}
    print("Local safe KG graphs built:",len(KG_LOCAL_GRAPHS))

In [ ]:
if PYG_AVAILABLE:
    class RGCNKGModel(nn.Module):
        def __init__(self,n_node_ids,n_node_types,n_rel,n_tasks,hidden=256,layers=3):
            super().__init__();self.idemb=nn.Embedding(max(n_node_ids,2),hidden);self.typeemb=nn.Embedding(n_node_types,hidden)
            self.convs=nn.ModuleList([RGCNConv(hidden,hidden,max(n_rel,1),num_bases=min(16,max(n_rel,1))) for _ in range(layers)])
            self.embed=nn.Linear(hidden,256);self.head=nn.Linear(256,n_tasks)
        def forward(self,d):
            x=self.idemb(d.node_id)+self.typeemb(d.node_type)
            for c in self.convs:x=torch.relu(c(x,d.edge_index,d.edge_type))
            # root node is first node of each graph; use ptr
            roots=d.ptr[:-1];z=torch.relu(self.embed(x[roots]));return self.head(z),z

    class GPSKGModel(nn.Module):
        """Local GINE + graph-level Transformer approximation of GPS on per-chemical mechanism graphs."""
        def __init__(self,n_node_ids,n_node_types,n_rel,n_tasks,hidden=256,layers=4,heads=8,dropout=.25):
            super().__init__();self.idemb=nn.Embedding(max(n_node_ids,2),hidden);self.typeemb=nn.Embedding(n_node_types,hidden);self.relemb=nn.Embedding(max(n_rel,1),hidden)
            self.layers=nn.ModuleList()
            for _ in range(layers):
                mlp=nn.Sequential(nn.Linear(hidden,hidden*2),nn.LeakyReLU(.1),nn.Linear(hidden*2,hidden))
                self.layers.append(GPSConv(hidden,GINEConv(mlp,edge_dim=hidden),heads=heads,dropout=dropout,attn_type="multihead"))
            self.norm=nn.LayerNorm(hidden);self.embed=nn.Linear(hidden,256);self.head=nn.Linear(256,n_tasks)
        def forward(self,d):
            x=self.idemb(d.node_id)+self.typeemb(d.node_type);ea=self.relemb(d.edge_type)
            for layer in self.layers:x=layer(x,d.edge_index,d.batch,edge_attr=ea)
            roots=d.ptr[:-1];z=torch.relu(self.embed(self.norm(x[roots])));return self.head(z),z

In [ ]:
def run_kg_branch_cv(name,builder):
    if not KG_LOCAL_GRAPHS:
        print(name,"skipped: safe KG local graphs unavailable");return None
    kg_train_local=np.where(np.isin(train_df.global_index.values,list(KG_LOCAL_GRAPHS.keys())))[0]
    kg_test_global=np.array([i for i in test_idx if i in KG_LOCAL_GRAPHS],dtype=int)
    if len(kg_test_global)==0:print(name,"skipped: no KG test overlap");return None
    # Artifacts use full train/test shapes; missing-KG rows remain NaN and are handled by fusion modality masks.
    oof=np.full((len(train_df),len(TARGET_COLS)),np.nan,np.float32);oofz=np.zeros((len(train_df),256),np.float32);tps=[];tzs=[]
    test_pos={g:i for i,g in enumerate(test_idx)}
    for fold in range(N_CV_FOLDS):
        trl=np.array([i for i in kg_train_local if train_df.cv_fold.iloc[i]!=fold]);val=np.array([i for i in kg_train_local if train_df.cv_fold.iloc[i]==fold])
        trg=train_df.global_index.values[trl];vag=train_df.global_index.values[val]
        model,best=train_graph_model(builder(),[KG_LOCAL_GRAPHS[i] for i in trg],[KG_LOCAL_GRAPHS[i] for i in vag],PRIMARY_SEED+fold,"asymmetric")
        vi,pv,zv=graph_predict(model,PyGDataLoader([KG_LOCAL_GRAPHS[i] for i in vag],batch_size=CFG["batch_size"],shuffle=False))
        ti,pt,zt=graph_predict(model,PyGDataLoader([KG_LOCAL_GRAPHS[i] for i in kg_test_global],batch_size=CFG["batch_size"],shuffle=False))
        vm={g:i for i,g in enumerate(vag)};ordv=np.argsort([vm[int(i)] for i in vi]);oof[val]=pv[ordv];oofz[val]=zv[ordv]
        fullpt=np.full((len(test_idx),len(TARGET_COLS)),np.nan,np.float32);fullzt=np.zeros((len(test_idx),256),np.float32)
        for ii,p,z in zip(ti,pt,zt):fullpt[test_pos[int(ii)]]=p;fullzt[test_pos[int(ii)]]=z
        tps.append(fullpt);tzs.append(fullzt);print(f"{name} fold {fold+1}: {best:.4f}")
    tp=np.nanmean(np.stack(tps),axis=0);tz=np.mean(tzs,axis=0)
    save_model_artifact(name,oof,tp,oofz,tz,{"type":"safe_KG","coverage_train":len(kg_train_local),"coverage_test":len(kg_test_global)})
    return oof,tp,oofz,tz

In [ ]:
if RUN_KG_MODELS and PYG_AVAILABLE and KG_LOCAL_GRAPHS:
    nids=max(KG_NODE_TOKEN_VOCAB.values(),default=1)+1;nrel=max(len(KG_RELATION_VOCAB),1)
    RGCN_RESULT=run_kg_branch_cv("RGCN_safe",lambda:RGCNKGModel(nids,4,nrel,len(TARGET_COLS),hidden=192 if LOW_MEMORY else 256,layers=3))
    GPS_RESULT=run_kg_branch_cv("GPS_safe",lambda:GPSKGModel(nids,4,nrel,len(TARGET_COLS),hidden=192 if LOW_MEMORY else 256,layers=3 if LOW_MEMORY else 4,heads=4 if LOW_MEMORY else 8))
else:
    print("Safe KG models skipped because KG files/graphs are unavailable or disabled.")

## 26. Global HGT on the leakage-safe heterogeneous graph

HGT preserves node and relation types in one global graph. Chemical, gene, pathway and other nodes receive separate learnable embeddings; matched chemical nodes also receive fold-local structural features. This section runs only when safe triples and mapping files are available.

In [ ]:
GLOBAL_HETERO=None;HGT_CHEM_TO_GLOBAL={}
if PYG_AVAILABLE and SAFE_TRIPLES is not None and KG_COMPOUND_DF is not None and kg_cid_col:
    src,rel,dst=TRIPLE_COLS["source"],TRIPLE_COLS["relation"],TRIPLE_COLS["target"]
    def id_set_from_file(path):
        if not path:return set()
        d=pd.read_csv(path);c=d.columns[0];return set(d[c].astype(str))
    gene_ids=id_set_from_file(KG_FILES.get("genes"));path_ids=id_set_from_file(KG_FILES.get("pathways"));chem_ids=set(KG_COMPOUND_DF[kg_cid_col].astype(str))
    def ntype(v):
        v=str(v)
        if v in chem_ids:return "chemical"
        if v in gene_ids:return "gene"
        if v in path_ids:return "pathway"
        return "other"
    node_sets=defaultdict(set);typed_edges=defaultdict(list)
    for _,r in SAFE_TRIPLES[[src,rel,dst]].iterrows():
        a,b,rr=str(r[src]),str(r[dst]),re.sub(r"[^A-Za-z0-9_]","_",str(r[rel]))[:80]
        ta,tb=ntype(a),ntype(b);node_sets[ta].add(a);node_sets[tb].add(b);typed_edges[(ta,rr,tb)].append((a,b))
    node_maps={t:{v:i for i,v in enumerate(sorted(vals))} for t,vals in node_sets.items()}
    hd=HeteroData()
    for t,mp in node_maps.items():hd[t].node_id=torch.arange(len(mp))
    for et,edges in typed_edges.items():
        a,r,b=et;hd[et].edge_index=torch.tensor([[node_maps[a][x] for x,_ in edges],[node_maps[b][y] for _,y in edges]],dtype=torch.long)
    # Map KG chemical nodes to curated global indices through canonical SMILES.
    cur_map={s:i for i,s in enumerate(curated_df.canonical_smiles)}
    for _,r in KG_COMPOUND_DF.iterrows():
        cid=str(r[kg_cid_col]);s=r.get("canonical_smiles")
        if s in cur_map and cid in node_maps.get("chemical",{}):HGT_CHEM_TO_GLOBAL[node_maps["chemical"][cid]]=cur_map[s]
    GLOBAL_HETERO=hd
    print("Global HeteroData:",hd.metadata(),"mapped chemicals:",len(HGT_CHEM_TO_GLOBAL))
else:print("Global HGT data unavailable.")

In [ ]:
if PYG_AVAILABLE:
    class GlobalHGTModel(nn.Module):
        def __init__(self,data,n_tasks,hidden=256,layers=3,heads=4):
            super().__init__();self.emb=nn.ModuleDict({t:nn.Embedding(data[t].num_nodes,hidden) for t in data.node_types});self.convs=nn.ModuleList([HGTConv(hidden,hidden,data.metadata(),heads=heads,group="sum") for _ in range(layers)]);self.proj=nn.Linear(hidden,256);self.head=nn.Linear(256,n_tasks)
        def forward(self,data):
            x={t:self.emb[t](data[t].node_id.to(next(self.parameters()).device)) for t in data.node_types}
            ei={k:v.to(next(self.parameters()).device) for k,v in data.edge_index_dict.items()}
            for c in self.convs:
                new_x=c(x,ei)
                x={k:torch.relu(new_x[k]) if k in new_x else x[k] for k in x}
            z=torch.relu(self.proj(x["chemical"]));return self.head(z),z

def run_global_hgt_cv():
    if GLOBAL_HETERO is None or len(HGT_CHEM_TO_GLOBAL)<100:return None
    hd=GLOBAL_HETERO.to(DEVICE);chem_nodes=np.array(sorted(HGT_CHEM_TO_GLOBAL));globals_=np.array([HGT_CHEM_TO_GLOBAL[i] for i in chem_nodes]);g_to_node={g:n for n,g in zip(chem_nodes,globals_)}
    oof=np.full((len(train_df),len(TARGET_COLS)),np.nan,np.float32);oofz=np.zeros((len(train_df),256),np.float32);tps=[];tzs=[];test_pos={g:i for i,g in enumerate(test_idx)}
    for fold in range(N_CV_FOLDS):
        tr_globals=set(train_df.loc[train_df.cv_fold!=fold,"global_index"]);va_globals=set(train_df.loc[train_df.cv_fold==fold,"global_index"])
        tr_nodes=np.array([g_to_node[g] for g in tr_globals if g in g_to_node],dtype=int)
        va_nodes=np.array([g_to_node[g] for g in va_globals if g in g_to_node],dtype=int)
        te_globals=np.array([g for g in test_idx if g in g_to_node],dtype=int)
        te_nodes=np.array([g_to_node[g] for g in te_globals],dtype=int)
        tr_nodes_t=torch.tensor(tr_nodes,dtype=torch.long,device=DEVICE)
        va_nodes_t=torch.tensor(va_nodes,dtype=torch.long,device=DEVICE)
        te_nodes_t=torch.tensor(te_nodes,dtype=torch.long,device=DEVICE)
        model=GlobalHGTModel(hd,len(TARGET_COLS),hidden=128 if LOW_MEMORY else 256,layers=2 if LOW_MEMORY else 3,heads=4).to(DEVICE)
        yt=torch.tensor(Y_FILLED[[HGT_CHEM_TO_GLOBAL[n] for n in tr_nodes]],device=DEVICE);mt=torch.tensor(M[[HGT_CHEM_TO_GLOBAL[n] for n in tr_nodes]],device=DEVICE);wp,wn=effective_num_weights(yt,mt);crit=MaskedAsymmetricLoss(wp,wn);opt=torch.optim.AdamW(model.parameters(),lr=5e-4,weight_decay=1e-5)
        best=-np.inf;state=None;wait=0
        for ep in range(CFG["epochs"]):
            model.train();opt.zero_grad();lg,z=model(hd);loss=crit(lg[tr_nodes_t],yt,mt);loss.backward();nn.utils.clip_grad_norm_(model.parameters(),1);opt.step()
            model.eval()
            with torch.no_grad():pv=torch.sigmoid(model(hd)[0][va_nodes_t]).cpu().numpy()
            vg=np.array([HGT_CHEM_TO_GLOBAL[n] for n in va_nodes]);auc=mean_masked_auc(Y_FILLED[vg],M[vg],pv)
            if auc>best+1e-5:best=auc;state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()};wait=0
            else:wait+=1
            if wait>=CFG["patience"]:break
        if state:model.load_state_dict(state)
        torch.save({"state_dict":model.state_dict(),"fold":fold,"metadata":hd.metadata()},DIRS["models"]/f"HGT_safe_fold{fold}.pt")
        model.eval()
        with torch.no_grad():lg,z=model(hd);pv=torch.sigmoid(lg[va_nodes_t]).cpu().numpy();zv=z[va_nodes_t].cpu().numpy();pt=torch.sigmoid(lg[te_nodes_t]).cpu().numpy();zt=z[te_nodes_t].cpu().numpy()
        local_map={g:i for i,g in enumerate(train_df.global_index.values)}
        for g,p,e in zip([HGT_CHEM_TO_GLOBAL[n] for n in va_nodes],pv,zv):oof[local_map[g]]=p;oofz[local_map[g]]=e
        ft=np.full((len(test_idx),len(TARGET_COLS)),np.nan,np.float32);fz=np.zeros((len(test_idx),256),np.float32)
        for g,p,e in zip(te_globals,pt,zt):ft[test_pos[g]]=p;fz[test_pos[g]]=e
        tps.append(ft);tzs.append(fz);print(f"HGT safe fold {fold+1}: {best:.4f}")
        del model;gc.collect();
        if torch.cuda.is_available():torch.cuda.empty_cache()
    tp=np.nanmean(np.stack(tps),0);tz=np.mean(tzs,0);save_model_artifact("HGT_safe",oof,tp,oofz,tz,{"type":"global_HGT","coverage":len(HGT_CHEM_TO_GLOBAL)});return oof,tp,oofz,tz

if RUN_KG_MODELS and GLOBAL_HETERO is not None:
    HGT_RESULT=run_global_hgt_cv()

## 26. Label-permutation leakage test

A small model is retrained on shuffled training labels. AUC should be approximately chance. Suspiciously high shuffled-label AUC triggers rejection and a feature/KG provenance audit.

In [ ]:
def label_permutation_test(max_train=1200,seed=13):
    rng=np.random.default_rng(seed);trg=train_df.global_index.values[:max_train]
    proc=FoldLocalFeatureProcessor(max_svd=64,random_state=seed);X=proc.fit_transform(X_FP[trg],X_DESC[trg])
    scores=[]
    for j in range(len(TARGET_COLS)):
        idx=M[trg,j].astype(bool)
        if idx.sum()<30 or len(np.unique(Y_FILLED[trg,j][idx]))<2:continue
        y=Y_FILLED[trg,j][idx].astype(int).copy();rng.shuffle(y)
        cut=int(.8*len(y));m=RandomForestClassifier(n_estimators=60,max_depth=6,random_state=seed,n_jobs=-1)
        m.fit(X[idx][:cut],y[:cut]);p=m.predict_proba(X[idx][cut:])[:,1]
        if len(np.unique(y[cut:]))==2:scores.append(roc_auc_score(y[cut:],p))
    return float(np.mean(scores)) if scores else np.nan

PERM_AUC=label_permutation_test()
print(f"Label permutation mean AUC: {PERM_AUC:.3f}")
if np.isfinite(PERM_AUC) and PERM_AUC>0.62:
    raise RuntimeError("Permutation AUC suspiciously high. Stop and audit leakage before modeling.")
else:
    print("Permutation test passed: performance is approximately chance.")

# Part V — Cross-modal pretraining and proposed multimodal architecture

## 27. Collect branch embeddings and modality masks

Only OOF embeddings are used to train fusion and ensemble weights. Final-test embeddings are never used for model selection.

In [ ]:
def available_embedding_artifacts():
    out={}
    for name in discover_artifacts():
        a=load_model_artifact(name)
        if "oof_embed" in a and "test_embed" in a:
            out[name]=a
    return out

EMBED_ARTIFACTS=available_embedding_artifacts()
print("Embedding branches available:",list(EMBED_ARTIFACTS))

In [ ]:
def select_core_modalities(arts):
    priority=["GPS_safe","AttentiveFP","MPNN","ChemBERTa","DeepTox_DNN","CapsNet"]
    selected={}
    for n in priority:
        if n in arts:selected[n]=arts[n]
    return selected

CORE_MODALITIES=select_core_modalities(EMBED_ARTIFACTS)
if len(CORE_MODALITIES)<2:
    print("Multimodal fusion-এর জন্য অন্তত দুইটি embedding branch প্রয়োজন। Available:",list(CORE_MODALITIES))
else:
    print("Fusion modalities:",list(CORE_MODALITIES))

## 28. Contrastive cross-modal alignment

Positive pairs are different views of the same molecule; negatives are other molecules. The fusion trainer uses cross-modal consistency/contrastive regularization on OOF embeddings only.

In [ ]:
def nt_xent(z1,z2,temp=0.1):
    z1=nn.functional.normalize(z1,dim=-1);z2=nn.functional.normalize(z2,dim=-1)
    logits=z1@z2.T/temp;labels=torch.arange(len(z1),device=z1.device)
    return .5*(nn.functional.cross_entropy(logits,labels)+nn.functional.cross_entropy(logits.T,labels))

class EndpointConditionedFusion(nn.Module):
    def __init__(self,input_dims:Dict[str,int],n_tasks:int,hidden=256,heads=8,layers=2,dropout=.25):
        super().__init__();self.names=list(input_dims);self.n_tasks=n_tasks;self.hidden=hidden
        self.proj=nn.ModuleDict({n:nn.Sequential(nn.Linear(d,hidden),nn.LayerNorm(hidden),nn.GELU()) for n,d in input_dims.items()})
        enc=nn.TransformerEncoderLayer(hidden,heads,hidden*4,dropout,batch_first=True,norm_first=True,activation="gelu")
        self.cross=nn.TransformerEncoder(enc,layers)
        self.task_emb=nn.Parameter(torch.randn(n_tasks,hidden)*.02)
        self.gate=nn.Sequential(nn.Linear(hidden*2,hidden),nn.GELU(),nn.Linear(hidden,1))
        self.shared=nn.Sequential(nn.Linear(hidden,hidden),nn.GELU(),nn.Dropout(dropout))
        self.nr=nn.Sequential(nn.Linear(hidden,hidden),nn.GELU(),nn.Dropout(dropout))
        self.sr=nn.Sequential(nn.Linear(hidden,hidden),nn.GELU(),nn.Dropout(dropout))
        self.heads=nn.ModuleList([nn.Linear(hidden,1) for _ in range(n_tasks)])
        self.modality_dropout=.12
    def forward(self,branches:Dict[str,torch.Tensor],modality_mask=None,return_gates=False):
        tokens=torch.stack([self.proj[n](branches[n]) for n in self.names],dim=1)
        B,K,H=tokens.shape
        if modality_mask is None:modality_mask=torch.ones(B,K,device=tokens.device)
        if self.training and self.modality_dropout>0:
            original_mask=modality_mask.clone()
            drop=(torch.rand(B,K,device=tokens.device)>self.modality_dropout).float()
            modality_mask=modality_mask*drop
            empty=modality_mask.sum(1)==0
            if empty.any():
                first_available=original_mask[empty].argmax(1)
                modality_mask[empty,first_available]=1
        tokens=self.cross(tokens,src_key_padding_mask=~modality_mask.bool())
        logits=[];allg=[]
        for t in range(self.n_tasks):
            q=self.task_emb[t].view(1,1,H).expand(B,K,H)
            s=self.gate(torch.cat([tokens,q],dim=-1)).squeeze(-1).masked_fill(~modality_mask.bool(),-1e9)
            a=torch.softmax(s,dim=1);z=(a.unsqueeze(-1)*tokens).sum(1);z=self.shared(z)
            z=self.nr(z) if TARGET_COLS[t].startswith("NR-") else self.sr(z)
            logits.append(self.heads[t](z));allg.append(a)
        logits=torch.cat(logits,dim=1);gates=torch.stack(allg,dim=1)
        return (logits,gates,tokens) if return_gates else logits

In [ ]:
def prepare_fusion_arrays(modalities):
    names=list(modalities); oof={};test={}; om=[];tm=[]
    for n in names:
        a=modalities[n];z1=a["oof_embed"].astype(np.float32);z2=a["test_embed"].astype(np.float32)
        valid1=np.isfinite(z1).all(1)&(np.abs(z1).sum(1)>0);valid2=np.isfinite(z2).all(1)&(np.abs(z2).sum(1)>0)
        z1=np.nan_to_num(z1);z2=np.nan_to_num(z2);oof[n]=z1;test[n]=z2;om.append(valid1);tm.append(valid2)
    return names,oof,test,np.stack(om,1).astype(np.float32),np.stack(tm,1).astype(np.float32)

if len(CORE_MODALITIES)>=2:
    FUSION_NAMES,FUSION_OOF_X,FUSION_TEST_X,FUSION_OOF_MASK,FUSION_TEST_MASK=prepare_fusion_arrays(CORE_MODALITIES)
    print({n:FUSION_OOF_X[n].shape for n in FUSION_NAMES})

## 29. Three-stage fusion training

- Stage A: branch models are trained separately and OOF embeddings are saved.
- Stage B: encoders remain frozen; cross-modal attention, gates and task heads are trained.
- Stage C: optional partial joint fine-tuning is supported through branch checkpoints; in low-memory mode the notebook keeps encoders frozen and fine-tunes fusion only.

In [ ]:
def train_fusion_fold(modalities,mod_masks,trl,val,seed=13,hidden=256,dropout=.25,contrastive_weight=.02):
    seed_everything(seed);dims={n:modalities[n].shape[1] for n in modalities}
    model=EndpointConditionedFusion(dims,len(TARGET_COLS),hidden=hidden,heads=4 if LOW_MEMORY else 8,layers=2,dropout=dropout).to(DEVICE)
    wpos,wneg=effective_num_weights(torch.tensor(Y_FILLED[train_df.global_index.values[trl]],device=DEVICE),torch.tensor(M[train_df.global_index.values[trl]],device=DEVICE))
    criterion=MaskedAsymmetricLoss(wpos,wneg)
    opt=torch.optim.AdamW(model.parameters(),lr=3e-4,weight_decay=1e-5);sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=CFG["epochs"])
    trds=TensorDataset(torch.tensor(trl),torch.tensor(Y_FILLED[train_df.global_index.values[trl]]),torch.tensor(M[train_df.global_index.values[trl]]))
    loader=DataLoader(trds,batch_size=CFG["batch_size"],shuffle=True)
    best=-np.inf;state=None;wait=0
    for ep in range(CFG["epochs"]):
        model.train()
        for ib,yb,mb in loader:
            ids=ib.numpy();bx={n:torch.tensor(modalities[n][ids],device=DEVICE) for n in modalities};mm=torch.tensor(mod_masks[ids],device=DEVICE);yb=yb.to(DEVICE);mb=mb.to(DEVICE)
            opt.zero_grad();logits,gates,toks=model(bx,mm,True);loss=criterion(logits,yb,mb)
            if len(modalities)>=2 and contrastive_weight>0:
                valid=(mm[:,0]>0)&(mm[:,1]>0)
                if valid.sum()>2:loss=loss+contrastive_weight*nt_xent(toks[valid,0],toks[valid,1])
            loss.backward();nn.utils.clip_grad_norm_(model.parameters(),1);opt.step()
        sched.step();model.eval()
        with torch.no_grad():
            bx={n:torch.tensor(modalities[n][val],device=DEVICE) for n in modalities};mm=torch.tensor(mod_masks[val],device=DEVICE);p=torch.sigmoid(model(bx,mm)).cpu().numpy()
        vg=train_df.global_index.values[val];auc=mean_masked_auc(Y_FILLED[vg],M[vg],p)
        if auc>best+1e-5:best=auc;state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()};wait=0
        else:wait+=1
        if wait>=CFG["patience"]:break
    if state:model.load_state_dict(state)
    return model,best

def fusion_predict(model,modalities,mod_mask,idx):
    model.eval();ps=[];gs=[]
    with torch.no_grad():
        for s in range(0,len(idx),256):
            ii=idx[s:s+256];bx={n:torch.tensor(modalities[n][ii],device=DEVICE) for n in modalities};mm=torch.tensor(mod_mask[ii],device=DEVICE)
            lg,g,_=model(bx,mm,True);ps.append(torch.sigmoid(lg).cpu().numpy());gs.append(g.cpu().numpy())
    return np.vstack(ps),np.vstack(gs)

In [ ]:
def run_fusion_cv(params=None, predict_test=True, artifact_name="Multimodal_Fusion", seed_offset=0):
    params=params or {};oof=np.full((len(train_df),len(TARGET_COLS)),np.nan,np.float32)
    tps=[];oog=np.zeros((len(train_df),len(TARGET_COLS),len(FUSION_NAMES)),np.float32);tgs=[]
    for fold in range(N_CV_FOLDS):
        trl=np.where(train_df.cv_fold.values!=fold)[0];val=np.where(train_df.cv_fold.values==fold)[0]
        model,best=train_fusion_fold(FUSION_OOF_X,FUSION_OOF_MASK,trl,val,PRIMARY_SEED+seed_offset+fold,**params)
        pv,gv=fusion_predict(model,FUSION_OOF_X,FUSION_OOF_MASK,val)
        oof[val]=pv;oog[val]=gv
        if predict_test:
            pt,gt=fusion_predict(model,FUSION_TEST_X,FUSION_TEST_MASK,np.arange(len(test_idx)));tps.append(pt);tgs.append(gt)
            torch.save({"state_dict":model.state_dict(),"fold":fold,"params":params,"modalities":FUSION_NAMES},DIRS["models"]/f"{artifact_name}_fold{fold}.pt")
        print(f"{artifact_name} fold {fold+1}: {best:.4f}")
        del model;gc.collect()
        if torch.cuda.is_available():torch.cuda.empty_cache()
    if not predict_test:
        return oof,None,oog,None
    tp=np.mean(tps,axis=0);tg=np.mean(tgs,axis=0)
    oof_embed=oog.reshape(len(oog),-1);test_embed=tg.reshape(len(tg),-1)
    save_model_artifact(artifact_name,oof,tp,oof_embed,test_embed,{"modalities":FUSION_NAMES,"cv":N_CV_FOLDS,"params":params,"seed_offset":seed_offset})
    np.savez_compressed(ARTIFACT_DIR/f"{artifact_name}_gates.npz",oof_gates=oog,test_gates=tg)
    return oof,tp,oog,tg

if RUN_MULTIMODAL_FUSION and len(CORE_MODALITIES)>=2:
    FUSION_OOF,FUSION_TEST,FUSION_OOF_GATES,FUSION_TEST_GATES=run_fusion_cv({"hidden":192 if LOW_MEMORY else 256,"dropout":.25,"contrastive_weight":.02})
else:
    print("Multimodal fusion skipped: at least two branch embeddings are required.")

## 30. Optuna HPO with robust OOF objective

The objective rewards high mean endpoint AUC while penalizing large endpoint variance, weak minimum endpoint performance and calibration error. The final test is never queried by Optuna.

In [ ]:
def robust_oof_objective(y,m,p):
    aucs=[];eces=[]
    for j in range(y.shape[1]):
        idx=m[:,j].astype(bool);yy=y[idx,j];pp=p[idx,j]
        if len(np.unique(yy))==2:
            aucs.append(roc_auc_score(yy,pp));eces.append(expected_calibration_error(yy,pp))
    if not aucs:return -1
    aucs=np.array(aucs)
    return float(aucs.mean()-.15*aucs.std()-.10*max(0,.82-aucs.min())-.05*np.mean(eces))

BEST_FUSION_PARAMS={"hidden":192 if LOW_MEMORY else 256,"dropout":.25,"contrastive_weight":.02}
if RUN_HPO and RUN_MULTIMODAL_FUSION and len(CORE_MODALITIES)>=2:
    import optuna
    def objective(trial):
        params={
            "hidden":trial.suggest_categorical("hidden",[192,256,384] if not LOW_MEMORY else [128,192]),
            "dropout":trial.suggest_float("dropout",.12,.42),
            "contrastive_weight":trial.suggest_float("contrastive_weight",0.0,.08),
        }
        oof,_,_,_=run_fusion_cv(params,predict_test=False,artifact_name="HPO_internal")
        return robust_oof_objective(Y_FILLED[train_df.global_index.values],M[train_df.global_index.values],oof)
    print("Optuna HPO শুরু হচ্ছে। Trial চলাকালে final test prediction generate বা evaluate হবে না।")
    study=optuna.create_study(direction="maximize",sampler=optuna.samplers.TPESampler(seed=PRIMARY_SEED),pruner=optuna.pruners.MedianPruner())
    study.optimize(objective,n_trials=CFG["hpo_trials"],show_progress_bar=True)
    BEST_FUSION_PARAMS=study.best_params
    with open(DIRS["outputs"]/"optuna_best_fusion.json","w") as f:json.dump({"value":study.best_value,"params":study.best_params},f,indent=2)
    print("Best robust OOF objective:",study.best_value,"params:",study.best_params)
    OPT_FUSION=run_fusion_cv(BEST_FUSION_PARAMS,predict_test=True,artifact_name="Multimodal_Fusion_Optimized")
else:
    print("Optuna HPO disabled or fusion unavailable.")

# Final-candidate seed ensemble after HPO freeze.
if RUN_MULTIMODAL_FUSION and len(CORE_MODALITIES)>=2 and len(CFG["seeds"])>1:
    seed_runs=[]
    for s in CFG["seeds"]:
        seed_runs.append(run_fusion_cv(BEST_FUSION_PARAMS,True,f"Fusion_seed{s}",seed_offset=s))
    so=np.mean([r[0] for r in seed_runs],axis=0);st=np.mean([r[1] for r in seed_runs],axis=0)
    sg_o=np.mean([r[2] for r in seed_runs],axis=0);sg_t=np.mean([r[3] for r in seed_runs],axis=0)
    save_model_artifact("Multimodal_Fusion_SeedEnsemble",so,st,sg_o.reshape(len(so),-1),sg_t.reshape(len(st),-1),{"seeds":CFG["seeds"],"params":BEST_FUSION_PARAMS})

# Part VI — Hybrid heads and endpoint-wise OOF ensemble

## 31. Hybrid deep-embedding classifiers

Candidates include GPS+AttentiveFP → SVM-RBF, ChemBERTa → SVM-RBF, CapsNet → SVM/RF and fusion embeddings → SVM-RBF. A hybrid is retained only if it improves OOF AUC or adds useful error diversity.

In [ ]:
def run_embedding_hybrid(name,branch_names,classifier="svm"):
    """Nested-on-OOF hybrid: fit on other CV folds and predict the held-out fold."""
    arts=[load_model_artifact(n) for n in branch_names]
    Xo=np.concatenate([a["oof_embed"] for a in arts],axis=1)
    Xt=np.concatenate([a["test_embed"] for a in arts],axis=1)
    valid_o=np.isfinite(Xo).all(1)&(np.abs(Xo).sum(1)>0)
    valid_t=np.isfinite(Xt).all(1)&(np.abs(Xt).sum(1)>0)
    oof=np.full((len(train_df),len(TARGET_COLS)),np.nan,np.float32)
    test_folds=[]
    y_all=Y_FILLED[train_df.global_index.values]
    m_all=M[train_df.global_index.values]
    for fold in range(N_CV_FOLDS):
        tr_fold=(train_df.cv_fold.values!=fold)&valid_o
        va_fold=(train_df.cv_fold.values==fold)&valid_o
        fold_test=np.full((len(test_idx),len(TARGET_COLS)),np.nan,np.float32)
        for j in range(len(TARGET_COLS)):
            tr=tr_fold&m_all[:,j].astype(bool)
            va=va_fold
            if tr.sum()<20 or len(np.unique(y_all[tr,j]))<2:continue
            sc=StandardScaler().fit(Xo[tr]);xtr=sc.transform(Xo[tr]);xva=sc.transform(Xo[va]);xte=sc.transform(Xt[valid_t])
            yj=y_all[tr,j].astype(int)
            if classifier=="svm":
                clf=SVC(C=4,gamma="scale",kernel="rbf",probability=False,class_weight="balanced",cache_size=4096,random_state=PRIMARY_SEED+fold+j)
            else:
                clf=RandomForestClassifier(n_estimators=350,class_weight="balanced_subsample",n_jobs=-1,random_state=PRIMARY_SEED+fold+j)
            clf.fit(xtr,yj)
            oof[np.where(va)[0],j]=probability_like(clf,xva)
            fold_test[valid_t,j]=probability_like(clf,xte)
        test_folds.append(fold_test)
    test=np.nanmean(np.stack(test_folds),axis=0)
    save_model_artifact(name,oof,test,metadata={"type":"hybrid_nested_oof","branches":branch_names,"classifier":classifier})
    return oof,test

if RUN_HYBRIDS:
    arts=discover_artifacts()
    if "GPS_safe" in arts and "AttentiveFP" in arts:run_embedding_hybrid("GPS_AttentiveFP_SVM",["GPS_safe","AttentiveFP"],"svm")
    if "ChemBERTa" in arts:run_embedding_hybrid("ChemBERTa_SVM",["ChemBERTa"],"svm")
    if "CapsNet" in arts:run_embedding_hybrid("CapsNet_SVM",["CapsNet"],"svm");run_embedding_hybrid("CapsNet_RF",["CapsNet"],"rf")
    if "Multimodal_Fusion" in arts:run_embedding_hybrid("Fusion_SVM",["Multimodal_Fusion"],"svm")

## 32. Endpoint-wise OOF-optimized ensemble

Weights are non-negative and sum to one. They are learned separately for each endpoint from OOF predictions only. The final test is predicted once after weights are frozen.

In [ ]:
def collect_prediction_artifacts():
    out={}
    for n in discover_artifacts():
        a=load_model_artifact(n)
        if "oof_prob" in a and "test_prob" in a:out[n]=a
    return out

PRED_ARTS=collect_prediction_artifacts()
print("Ensemble candidates:",list(PRED_ARTS))

In [ ]:
def optimize_blend_weights(y,m,pred_dict,variance_penalty=.002):
    names=list(pred_dict);K=len(names);T=y.shape[1];W=np.zeros((T,K),float)
    for j in range(T):
        cols=[];kept=[]
        for k,n in enumerate(names):
            p=pred_dict[n]["oof_prob"][:,j];
            if np.isfinite(p).sum()>20:cols.append(p);kept.append(k)
        if not cols:continue
        P=np.stack(cols,1);valid=m[:,j].astype(bool)&np.isfinite(P).all(1);yy=y[valid,j]
        if valid.sum()<20 or len(np.unique(yy))<2:continue
        PP=P[valid]
        def obj(w):
            blend=PP@w;auc=roc_auc_score(yy,blend);return -auc+variance_penalty*np.var(w)
        x0=np.ones(len(kept))/len(kept);cons={"type":"eq","fun":lambda w:w.sum()-1};bounds=[(0,1)]*len(kept)
        res=minimize(obj,x0,bounds=bounds,constraints=cons,method="SLSQP",options={"maxiter":200})
        w=res.x if res.success else x0
        for kk,v in zip(kept,w):W[j,kk]=v
    return names,W

def blend_predictions(pred_dict,names,W,key):
    N=next(iter(pred_dict.values()))[key].shape[0];T=W.shape[0];out=np.zeros((N,T),float)
    for j in range(T):
        num=np.zeros(N);den=np.zeros(N)
        for k,n in enumerate(names):
            p=pred_dict[n][key][:,j];v=np.isfinite(p);num[v]+=W[j,k]*p[v];den[v]+=W[j,k]
        out[:,j]=np.divide(num,den,out=np.full(N,.5),where=den>0)
    return out

Y_TRAIN=Y_FILLED[train_df.global_index.values];M_TRAIN=M[train_df.global_index.values]
ENSEMBLE_NAMES,ENSEMBLE_W=optimize_blend_weights(Y_TRAIN,M_TRAIN,PRED_ARTS)
ENSEMBLE_OOF=blend_predictions(PRED_ARTS,ENSEMBLE_NAMES,ENSEMBLE_W,"oof_prob")
ENSEMBLE_TEST=blend_predictions(PRED_ARTS,ENSEMBLE_NAMES,ENSEMBLE_W,"test_prob")
np.savez_compressed(DIRS["outputs"]/"final_ensemble.npz",oof=ENSEMBLE_OOF,test=ENSEMBLE_TEST,weights=ENSEMBLE_W,names=np.array(ENSEMBLE_NAMES))
print("OOF ensemble weights frozen. Final test was not used for weight learning.")

In [ ]:
# Ensemble weight heatmap
plt.figure(figsize=(max(10,len(ENSEMBLE_NAMES)*1.1),7))
sns.heatmap(pd.DataFrame(ENSEMBLE_W,index=TARGET_COLS,columns=ENSEMBLE_NAMES),annot=True,fmt=".2f",cmap="viridis")
plt.title("Endpoint-wise OOF-Optimized Ensemble Weights",fontweight="bold")
plt.tight_layout();plt.savefig(DIRS["figures"]/"10_ensemble_weights.png",dpi=220,bbox_inches="tight");plt.show()

# Part VII — Frozen evaluation, uncertainty, calibration and trustworthy outputs

## 33. OOF threshold selection and one-time final test evaluation

ROC-AUC uses probabilities and no threshold. Accuracy/F1/BAC thresholds are chosen per endpoint from OOF predictions only and then frozen.

In [ ]:
OOF_THRESHOLDS=thresholds_from_oof(Y_TRAIN,M_TRAIN,ENSEMBLE_OOF,"balanced_accuracy")
TEST_Y=Y_FILLED[test_idx];TEST_M=M[test_idx]
OOF_METRICS=evaluate_multitask(Y_TRAIN,M_TRAIN,ENSEMBLE_OOF,OOF_THRESHOLDS,TARGET_COLS)
TEST_METRICS=evaluate_multitask(TEST_Y,TEST_M,ENSEMBLE_TEST,OOF_THRESHOLDS,TARGET_COLS)
OOF_SUMMARY=macro_summary(OOF_METRICS);TEST_SUMMARY=macro_summary(TEST_METRICS)

print("OOF summary:", {k:round(v,4) for k,v in OOF_SUMMARY.items()})
print("Frozen test summary:", {k:round(v,4) for k,v in TEST_SUMMARY.items()})
display(TEST_METRICS.round(4))
OOF_METRICS.to_csv(DIRS["outputs"]/"final_oof_per_endpoint_metrics.csv",index=False)
TEST_METRICS.to_csv(DIRS["outputs"]/"final_test_per_endpoint_metrics.csv",index=False)

In [ ]:
# Bootstrap confidence interval for final macro mean AUC
ci=bootstrap_macro_auc(TEST_Y,TEST_M,ENSEMBLE_TEST,n_boot=200 if EXECUTION_MODE=="QUICK" else 2000,seed=PRIMARY_SEED)
print(f"Final macro mean ROC-AUC = {TEST_SUMMARY['Mean_AUC']:.4f}")
print(f"Bootstrap 95% CI = [{ci[0]:.4f}, {ci[2]:.4f}]")
print("Target status:", "ACHIEVED" if TEST_SUMMARY["Mean_AUC"]>=TARGET_MEAN_AUC else "NOT YET ACHIEVED — report honestly and inspect bottlenecks")

## 34. Model leaderboard and endpoint heatmap

In [ ]:
leader=[];heat={}
for n,a in PRED_ARTS.items():
    # Use OOF for model selection table; test only for final contextual reporting after freeze.
    o=evaluate_multitask(Y_TRAIN,M_TRAIN,a["oof_prob"],None,TARGET_COLS)
    t=evaluate_multitask(TEST_Y,TEST_M,a["test_prob"],OOF_THRESHOLDS,TARGET_COLS)
    leader.append({"Model":n,"OOF Mean AUC":o.AUC.mean(),"Test Mean AUC":t.AUC.mean(),"Test Accuracy":t.Accuracy.mean()})
    heat[n]=t.set_index("Endpoint")["AUC"]
leader.append({"Model":"Final Ensemble","OOF Mean AUC":OOF_METRICS.AUC.mean(),"Test Mean AUC":TEST_METRICS.AUC.mean(),"Test Accuracy":TEST_METRICS.Accuracy.mean()})
LEADERBOARD=pd.DataFrame(leader).sort_values("Test Mean AUC",ascending=False)
display(LEADERBOARD.round(4))
LEADERBOARD.to_csv(DIRS["outputs"]/"model_leaderboard.csv",index=False)

In [ ]:
plt.figure(figsize=(12,max(5,len(LEADERBOARD)*.42)))
plot=LEADERBOARD.sort_values("Test Mean AUC")
plt.barh(plot["Model"],plot["Test Mean AUC"])
plt.axvline(TARGET_MEAN_AUC,linestyle="--",label=f"Stretch target {TARGET_MEAN_AUC:.2f}")
for i,v in enumerate(plot["Test Mean AUC"]):plt.text(v+.002,i,f"{v:.3f}",va="center")
plt.xlim(max(0.5,plot["Test Mean AUC"].min()-.05),1.0);plt.xlabel("Frozen Test Macro Mean ROC-AUC");plt.title("Model Comparison",fontweight="bold");plt.legend();plt.tight_layout()
plt.savefig(DIRS["figures"]/"11_model_comparison.png",dpi=220,bbox_inches="tight");plt.show()

In [ ]:
heat_df=pd.DataFrame(heat).reindex(TARGET_COLS)
heat_df["Final Ensemble"]=TEST_METRICS.set_index("Endpoint")["AUC"]
plt.figure(figsize=(max(10,len(heat_df.columns)*1.0),7))
sns.heatmap(heat_df,annot=True,fmt=".2f",vmin=.5,vmax=1,cmap="RdYlGn")
plt.title("AUC Heatmap — Models × Tox21 Endpoints",fontweight="bold");plt.tight_layout();plt.savefig(DIRS["figures"]/"12_auc_heatmap.png",dpi=220,bbox_inches="tight");plt.show()

## 35. Per-endpoint ROC curves and best-vs-hardest confusion matrices

In [ ]:
fig,axes=plt.subplots(3,4,figsize=(16,12));axes=axes.ravel()
for j,(ax,t) in enumerate(zip(axes,TARGET_COLS)):
    idx=TEST_M[:,j].astype(bool);y=TEST_Y[idx,j];p=ENSEMBLE_TEST[idx,j]
    if len(np.unique(y))==2:
        fpr,tpr,_=roc_curve(y,p);auc=roc_auc_score(y,p);ax.plot(fpr,tpr,label=f"AUC={auc:.3f}")
    ax.plot([0,1],[0,1],"--",linewidth=1);ax.set_title(t);ax.legend(loc="lower right");ax.set_xlabel("FPR");ax.set_ylabel("TPR")
plt.suptitle("Final Ensemble — ROC Curves",fontsize=16,fontweight="bold");plt.tight_layout();plt.savefig(DIRS["figures"]/"13_roc_curves.png",dpi=220,bbox_inches="tight");plt.show()

In [ ]:
best_t=TEST_METRICS.sort_values("AUC",ascending=False).Endpoint.iloc[0]
hard_t=TEST_METRICS.sort_values("AUC").Endpoint.iloc[0]
fig,axes=plt.subplots(1,2,figsize=(12,5))
for ax,t in zip(axes,[best_t,hard_t]):
    j=TARGET_COLS.index(t);idx=TEST_M[:,j].astype(bool);y=TEST_Y[idx,j].astype(int);pred=(ENSEMBLE_TEST[idx,j]>=OOF_THRESHOLDS[j]).astype(int)
    cm=confusion_matrix(y,pred,labels=[0,1]);sns.heatmap(cm,annot=True,fmt="d",cmap="Blues",ax=ax,cbar=False)
    ax.set_title(f"{t}\nAUC={safe_auc(y,ENSEMBLE_TEST[idx,j]):.3f}");ax.set_xlabel("Predicted");ax.set_ylabel("True");ax.set_xticklabels(["Non-Toxic","Toxic"]);ax.set_yticklabels(["Non-Toxic","Toxic"],rotation=0)
plt.suptitle("Confusion Matrices — Best vs. Hardest Endpoint",fontsize=15,fontweight="bold");plt.tight_layout();plt.savefig(DIRS["figures"]/"14_confusion_best_hardest.png",dpi=220,bbox_inches="tight");plt.show()

## 36. Probability calibration

Temperature scaling is fitted on OOF predictions only. Calibrated and uncalibrated probabilities, Brier score, ECE and reliability diagrams are reported.

In [ ]:
def fit_temperature(y,p):
    eps=1e-6;l=logit(np.clip(p,eps,1-eps))
    def nll(x):
        pp=expit(l/max(float(x[0]),1e-3));return -np.mean(y*np.log(pp+eps)+(1-y)*np.log(1-pp+eps))
    r=minimize(nll,[1.0],bounds=[(.05,10)],method="L-BFGS-B");return float(r.x[0])

def calibrate_probs(y,m,oof,test):
    temps=np.ones(y.shape[1]);out=test.copy()
    for j in range(y.shape[1]):
        idx=m[:,j].astype(bool);yy=y[idx,j];pp=oof[idx,j]
        if len(np.unique(yy))==2:
            temps[j]=fit_temperature(yy,pp);out[:,j]=expit(logit(np.clip(test[:,j],1e-6,1-1e-6))/temps[j])
    return temps,out

TEMPERATURES,CAL_TEST=calibrate_probs(Y_TRAIN,M_TRAIN,ENSEMBLE_OOF,ENSEMBLE_TEST)
cal_rows=[]
for j,t in enumerate(TARGET_COLS):
    idx=TEST_M[:,j].astype(bool);y=TEST_Y[idx,j];p=ENSEMBLE_TEST[idx,j];pc=CAL_TEST[idx,j]
    cal_rows.append({"Endpoint":t,"Temperature":TEMPERATURES[j],"Brier_raw":brier_score_loss(y,p),"Brier_cal":brier_score_loss(y,pc),"ECE_raw":expected_calibration_error(y,p),"ECE_cal":expected_calibration_error(y,pc)})
CAL_TABLE=pd.DataFrame(cal_rows);display(CAL_TABLE.round(4));CAL_TABLE.to_csv(DIRS["outputs"]/"calibration_metrics.csv",index=False)

In [ ]:
# Reliability diagrams for 4 representative endpoints
sel=TEST_METRICS.sort_values("AUC").Endpoint.iloc[[0,3,8,11]].tolist() if len(TEST_METRICS)>=12 else TARGET_COLS[:4]
fig,axes=plt.subplots(2,2,figsize=(11,9))
for ax,t in zip(axes.ravel(),sel):
    j=TARGET_COLS.index(t);idx=TEST_M[:,j].astype(bool);y=TEST_Y[idx,j];
    for p,label in [(ENSEMBLE_TEST[idx,j],"Raw"),(CAL_TEST[idx,j],"Calibrated")]:
        xs=[];ys=[]
        for lo,hi in zip(np.linspace(0,1,11)[:-1],np.linspace(0,1,11)[1:]):
            mm=(p>=lo)&(p<(hi if hi<1 else hi+1e-8))
            if mm.any():xs.append(p[mm].mean());ys.append(y[mm].mean())
        ax.plot(xs,ys,marker="o",label=label)
    ax.plot([0,1],[0,1],"--");ax.set_title(t);ax.set_xlabel("Predicted probability");ax.set_ylabel("Observed frequency");ax.legend()
plt.suptitle("Reliability Diagrams",fontweight="bold");plt.tight_layout();plt.savefig(DIRS["figures"]/"15_reliability.png",dpi=220,bbox_inches="tight");plt.show()

## 37. Applicability domain (AD)

Reliability combines maximum ECFP4 Tanimoto similarity to training molecules, scaffold novelty, multimodal embedding distance and KG coverage.

In [ ]:
# ECFP4 bit block occupies the first 2048 columns.
FP_BITS=(X_FP[:,:2048]>0).astype(np.uint8)
train_bits=FP_BITS[train_idx];test_bits=FP_BITS[test_idx]

def max_tanimoto_to_train(test_arr,train_arr,chunk=128):
    # Binary Tanimoto using matrix multiplication.
    tr=train_arr.astype(np.float32);tr_sum=tr.sum(1)
    out=[]
    for i in range(0,len(test_arr),chunk):
        q=test_arr[i:i+chunk].astype(np.float32);inter=q@tr.T;union=q.sum(1,keepdims=True)+tr_sum[None,:]-inter
        out.append(np.max(inter/np.maximum(union,1e-8),axis=1))
    return np.concatenate(out)

MAX_TANIMOTO=max_tanimoto_to_train(test_bits,train_bits)
SCAFFOLD_NOVEL=~curated_df.loc[test_idx,"scaffold"].isin(set(curated_df.loc[train_idx,"scaffold"])).values
# Use fusion embedding if available, otherwise DeepTox/AttentiveFP.
embed_name="Multimodal_Fusion" if "Multimodal_Fusion" in discover_artifacts() else next((n for n in ["AttentiveFP","DeepTox_DNN","ChemBERTa"] if n in discover_artifacts()),None)
if embed_name:
    a=load_model_artifact(embed_name);oe=a["oof_embed"];te=a["test_embed"]
    valid=np.isfinite(oe).all(1);nnm=NearestNeighbors(n_neighbors=min(5,valid.sum())).fit(StandardScaler().fit_transform(oe[valid]))
    # Fit scaler on OOF train embeddings only.
    sc=StandardScaler().fit(oe[valid]);dist,_=nnm.kneighbors(sc.transform(np.nan_to_num(te)));EMBED_DIST=dist.mean(1)
else:EMBED_DIST=np.full(len(test_idx),np.nan)
KG_COV_TEST=KG_COVERAGE[test_idx]
AD_TABLE=pd.DataFrame({"global_index":test_idx,"canonical_smiles":curated_df.loc[test_idx,"canonical_smiles"].values,"max_tanimoto":MAX_TANIMOTO,"scaffold_novel":SCAFFOLD_NOVEL,"embedding_distance":EMBED_DIST,"kg_covered":KG_COV_TEST})
q95=np.nanquantile(EMBED_DIST,.95) if np.isfinite(EMBED_DIST).any() else np.inf
AD_TABLE["inside_domain"]=(AD_TABLE.max_tanimoto>=.35)&(AD_TABLE.embedding_distance<=q95)
AD_TABLE.to_csv(DIRS["outputs"]/"applicability_domain.csv",index=False)
display(AD_TABLE.head())

In [ ]:
plt.figure(figsize=(10,5));plt.hist(AD_TABLE.max_tanimoto,bins=30,alpha=.85);plt.axvline(.35,linestyle="--",label="AD similarity threshold");plt.xlabel("Maximum Tanimoto similarity to training set");plt.ylabel("Test molecules");plt.title("Applicability Domain — Chemical Similarity",fontweight="bold");plt.legend();plt.tight_layout();plt.savefig(DIRS["figures"]/"16_applicability_domain.png",dpi=220,bbox_inches="tight");plt.show()

## 38. Mondrian split-conformal prediction sets

OOF nonconformity scores are calibrated separately by endpoint and class. Output sets may be `{Toxic}`, `{Non-toxic}`, `{Toxic, Non-toxic}` (uncertain) or empty (review/OOD).

In [ ]:
def conformal_sets_binary(y_cal,p_cal,p_test,alpha=.1):
    scores0=1-p_cal[y_cal==0];scores1=p_cal[y_cal==1]  # confidence of true class transformed below
    # Nonconformity = 1 - probability assigned to true class
    nc0=p_cal[y_cal==0];nc1=1-p_cal[y_cal==1]
    q0=np.quantile(nc0,min(1,math.ceil((len(nc0)+1)*(1-alpha))/max(len(nc0),1))) if len(nc0) else 1
    q1=np.quantile(nc1,min(1,math.ceil((len(nc1)+1)*(1-alpha))/max(len(nc1),1))) if len(nc1) else 1
    include0=p_test<=q0;include1=(1-p_test)<=q1
    return include0,include1,q0,q1

CONF_ROWS=[]
for j,t in enumerate(TARGET_COLS):
    idx=M_TRAIN[:,j].astype(bool);y=Y_TRAIN[idx,j].astype(int);p=ENSEMBLE_OOF[idx,j]
    i0,i1,q0,q1=conformal_sets_binary(y,p,CAL_TEST[:,j],alpha=.1)
    for r,(a,b) in enumerate(zip(i0,i1)):
        label="{Non-toxic,Toxic}" if a and b else "{Non-toxic}" if a else "{Toxic}" if b else "{}"
        CONF_ROWS.append({"test_row":r,"Endpoint":t,"prediction_set":label,"q_non_toxic":q0,"q_toxic":q1})
CONFORMAL=pd.DataFrame(CONF_ROWS);CONFORMAL.to_csv(DIRS["outputs"]/"conformal_prediction_sets.csv",index=False)
display(CONFORMAL.prediction_set.value_counts(normalize=True).rename("Fraction").to_frame())

# Part VIII — Explainability, fidelity, ablation and robustness

## 39. Molecular visualization and atom-level explanation hook

The notebook provides Integrated Gradients/GNNExplainer-compatible hooks for graph models and a perturbation-based atom deletion fidelity test. A heatmap alone is not accepted as proof.

In [ ]:
# Display a standardized molecule example
sample_i=int(test_idx[0]);mol=Chem.MolFromSmiles(curated_df.loc[sample_i,"canonical_smiles"])
display(Draw.MolToImage(mol,size=(420,300)))
print("Molecule:",curated_df.loc[sample_i,"canonical_smiles"])

In [ ]:
def atom_deletion_smiles(smiles,atom_indices):
    mol=Chem.RWMol(Chem.MolFromSmiles(smiles))
    for i in sorted(atom_indices,reverse=True):
        if i<mol.GetNumAtoms():mol.RemoveAtom(int(i))
    try:
        Chem.SanitizeMol(mol);return Chem.MolToSmiles(mol)
    except:return None

def atom_deletion_fidelity(predict_smiles_fn,smiles,importance,top_k=3,n_random=20,seed=13):
    base=float(predict_smiles_fn([smiles])[0]);order=np.argsort(importance)[::-1];top=order[:top_k]
    s_top=atom_deletion_smiles(smiles,top);top_p=float(predict_smiles_fn([s_top])[0]) if s_top else np.nan
    rng=np.random.default_rng(seed);rp=[]
    for _ in range(n_random):
        ids=rng.choice(len(importance),size=min(top_k,len(importance)),replace=False);ss=atom_deletion_smiles(smiles,ids)
        if ss:rp.append(float(predict_smiles_fn([ss])[0]))
    return {"base":base,"top_removed":top_p,"top_drop":base-top_p if np.isfinite(top_p) else np.nan,"random_drop_mean":base-np.mean(rp) if rp else np.nan}

print("Fidelity helper ready. Connect predict_smiles_fn to the selected final molecular branch.")

## 40. Fusion gate explanations

The gate heatmap shows which modality each endpoint used. It is interpreted together with perturbation-based modality deletion, not as standalone proof.

In [ ]:
gate_candidates=[ARTIFACT_DIR/"Multimodal_Fusion_Optimized_gates.npz",ARTIFACT_DIR/"Multimodal_Fusion_gates.npz"]
gate_path=next((p for p in gate_candidates if p.exists()),None)
if gate_path:
    gates=np.load(gate_path)["test_gates"].mean(0)
    plt.figure(figsize=(max(8,len(FUSION_NAMES)*1.4),7));sns.heatmap(pd.DataFrame(gates,index=TARGET_COLS,columns=FUSION_NAMES),annot=True,fmt=".2f",cmap="mako")
    plt.title("Mean Endpoint-Conditioned Modality Gates",fontweight="bold");plt.tight_layout();plt.savefig(DIRS["figures"]/"17_fusion_gates.png",dpi=220,bbox_inches="tight");plt.show()
else:print("Fusion gate artifact unavailable.")

## 41. Ablation study from available OOF branch predictions

Ablations test fingerprint only, graph only, KG only, ChemBERTa only, selected modality combinations and full ensemble. Only combinations with complete OOF predictions are reported.

In [ ]:
def simple_equal_blend(names):
    avail=[PRED_ARTS[n] for n in names if n in PRED_ARTS]
    if not avail:return None,None
    po=np.nanmean(np.stack([a["oof_prob"] for a in avail]),axis=0);pt=np.nanmean(np.stack([a["test_prob"] for a in avail]),axis=0)
    return po,pt

ABLATIONS={
    "Fingerprint DNN only":["DeepTox_DNN"],
    "Molecular graph only":["AttentiveFP"],
    "KG GPS only":["GPS_safe"],
    "ChemBERTa only":["ChemBERTa"],
    "Graph + KG":["AttentiveFP","GPS_safe"],
    "Graph + ChemBERTa":["AttentiveFP","ChemBERTa"],
    "KG + ChemBERTa":["GPS_safe","ChemBERTa"],
    "Full fusion":["Multimodal_Fusion"],
}
abl=[]
for label,names in ABLATIONS.items():
    po,pt=simple_equal_blend(names)
    if po is not None:
        abl.append({"Ablation":label,"Members":" + ".join([n for n in names if n in PRED_ARTS]),"OOF_AUC":evaluate_multitask(Y_TRAIN,M_TRAIN,po,None,TARGET_COLS).AUC.mean(),"Test_AUC":evaluate_multitask(TEST_Y,TEST_M,pt,OOF_THRESHOLDS,TARGET_COLS).AUC.mean()})
abl.append({"Ablation":"Full model + optimized ensemble","Members":", ".join(ENSEMBLE_NAMES),"OOF_AUC":OOF_METRICS.AUC.mean(),"Test_AUC":TEST_METRICS.AUC.mean()})
ABLATION_TABLE=pd.DataFrame(abl).sort_values("OOF_AUC",ascending=False);display(ABLATION_TABLE.round(4));ABLATION_TABLE.to_csv(DIRS["outputs"]/"ablation_results.csv",index=False)

## 42. Robustness checklist

The complete research run should repeat top candidates across three seeds, rerun top two with confirmatory 5-fold CV if compute permits, test alternative scaffold assignments, simulate missing KG, apply KG edge dropout, assess label-noise sensitivity and perform external validation after overlap removal.

In [ ]:
ROBUSTNESS_CHECKLIST=pd.DataFrame([
    ["Three final seeds",len(CFG["seeds"])>=3,"Run top candidates with seeds 13, 29, 47"],
    ["Alternative scaffold assignment",False,"Generate v2 manifest without touching frozen v1 result"],
    ["KG edge dropout",False,"Repeat GPS with 5%, 10%, 20% safe-edge dropout"],
    ["Missing-KG simulation",False,"Drop KG modality for random molecules; use modality mask"],
    ["Label-noise sensitivity",False,"Inject 1%, 3%, 5% train-only label noise"],
    ["Rare-endpoint stress test",False,"Report support and uncertainty per endpoint"],
    ["External validation",False,"Use ClinTox/ToxCast after overlap removal"],
],columns=["Test","Completed","Required action"])
display(ROBUSTNESS_CHECKLIST)
ROBUSTNESS_CHECKLIST.to_csv(DIRS["outputs"]/"robustness_checklist.csv",index=False)

# Part IX — Final structured report and reproducibility package

## 43. Final acceptance checklist and model card

In [ ]:
CHECKLIST={
    "Chemical curation audit complete":CURATED_CSV.exists(),
    "Duplicate conflict policy applied":"conflicting_endpoints" in curated_df,
    "Per-task mask validated":bool(np.all((M==0)|(M==1))),
    "Scaffold test frozen":SPLIT_FILE.exists(),
    "Zero molecule/scaffold leakage":len(train_ik&test_ik)==0 and len(train_scaffolds&test_scaffolds)==0,
    "Target KG edges removed":SAFE_TRIPLES is None or ("REMOVED_TRIPLES" in globals()),
    "Feature provenance recorded":(DIRS["outputs"]/"leakage_audit.json").exists(),
    "Branch baselines reproduced":len(discover_artifacts())>0,
    "GPS/HGT same-cohort comparison complete":"GPS_safe" in discover_artifacts(),
    "Fusion ablation complete":(DIRS["outputs"]/"ablation_results.csv").exists(),
    "OOF ensemble weights frozen":(DIRS["outputs"]/"final_ensemble.npz").exists(),
    "Final test evaluated once":(DIRS["outputs"]/"final_test_per_endpoint_metrics.csv").exists(),
    "95% CI reported":True,
    "AUC + Accuracy tables ready":True,
    "PR-AUC/BAC/F1 diagnostics ready":True,
    "Calibration/AD/conformal outputs ready":all((DIRS["outputs"]/f).exists() for f in ["calibration_metrics.csv","applicability_domain.csv","conformal_prediction_sets.csv"]),
}
CHECK_DF=pd.DataFrame({"Item":list(CHECKLIST),"Completed":list(CHECKLIST.values())})
display(CHECK_DF)
CHECK_DF.to_csv(DIRS["outputs"]/"final_acceptance_checklist.csv",index=False)

In [ ]:
MODEL_CARD={
    "model_name":"Leakage-Safe Knowledge-Augmented Multimodal Tox21 Ensemble",
    "primary_target":"Macro mean ROC-AUC across 12 endpoints",
    "stretch_target":TARGET_MEAN_AUC,
    "split":"75/25 Bemis-Murcko scaffold holdout; 3-fold scaffold-aware CV inside train",
    "missing_labels":"Per-task mask; no missing label imputation",
    "leakage_controls":["target assay outcome edge removal","feature provenance audit","label permutation test","zero scaffold overlap","no test tuning"],
    "final_metrics":TEST_SUMMARY,
    "bootstrap_auc_ci_95":[float(ci[0]),float(ci[2])],
    "limitations":["KG coverage may be incomplete","0.96 is not guaranteed","literature comparisons require matched protocols","external validation remains required"],
    "artifacts":discover_artifacts(),
}
with open(DIRS["reports"]/"model_card.json","w",encoding="utf-8") as f:json.dump(MODEL_CARD,f,indent=2,ensure_ascii=False)
print("Model card saved:",DIRS["reports"]/"model_card.json")

## 44. Final result dashboard

In [ ]:
# A compact final dashboard
summary_table=pd.DataFrame({
    "Metric":["OOF Macro AUC","Frozen Test Macro AUC","Frozen Test Accuracy","Frozen Test PR-AUC","Frozen Test BAC","95% CI lower","95% CI upper","Target"],
    "Value":[OOF_SUMMARY["Mean_AUC"],TEST_SUMMARY["Mean_AUC"],TEST_SUMMARY["Mean_Accuracy"],TEST_SUMMARY["Mean_PR_AUC"],TEST_SUMMARY["Mean_Balanced_Accuracy"],ci[0],ci[2],TARGET_MEAN_AUC]
})
display(summary_table.style.format({"Value":"{:.4f}"}).background_gradient(subset=["Value"],cmap="YlGn"))

status="TARGET ACHIEVED" if TEST_SUMMARY["Mean_AUC"]>=TARGET_MEAN_AUC else "BEST VALID RESULT RECORDED — TARGET NOT YET ACHIEVED"
color="#1A9B76" if TEST_SUMMARY["Mean_AUC"]>=TARGET_MEAN_AUC else "#E6A700"
display(HTML(f"<div class='research-card' style='border-left-color:{color}'><h3>{status}</h3><b>Frozen Test Macro ROC-AUC:</b> {TEST_SUMMARY['Mean_AUC']:.4f}<br><b>95% CI:</b> [{ci[0]:.4f}, {ci[2]:.4f}]<br><b>Scientific rule:</b> do not retune on this test result.</div>"))

## 45. Reproducibility manifest

This final cell records hashes, package versions, split files, model artifacts, predictions, metrics, figures and configuration.

In [ ]:
def sha256_file(path,chunk=1<<20):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        while True:
            b=f.read(chunk)
            if not b:break
            h.update(b)
    return h.hexdigest()

manifest={
    "timestamp":pd.Timestamp.utcnow().isoformat(),
    "dataset_path":str(TOX21_PATH),
    "dataset_sha256":sha256_file(TOX21_PATH),
    "curated_sha256":sha256_file(CURATED_CSV),
    "split_sha256":sha256_file(SPLIT_FILE),
    "cv_sha256":sha256_file(CV_FILE),
    "config":RUN_CONFIG,
    "model_artifacts":discover_artifacts(),
    "figures":[p.name for p in sorted(DIRS["figures"].glob("*.png"))],
    "outputs":[p.name for p in sorted(DIRS["outputs"].glob("*")) if p.is_file()],
    "package_versions":{"numpy":np.__version__,"pandas":pd.__version__,"torch":torch.__version__},
}
with open(DIRS["reports"]/"reproducibility_manifest.json","w",encoding="utf-8") as f:json.dump(manifest,f,indent=2,ensure_ascii=False)
print("Reproducibility manifest saved:",DIRS["reports"]/"reproducibility_manifest.json")
print("গবেষণা pipeline সম্পন্ন। outputs/, figures/, models/ এবং reports/ ফোল্ডার দেখুন।")

# End of notebook

### Interpretation rule

A macro mean test ROC-AUC ≥ 0.96 is accepted only when it is obtained on the single frozen scaffold test set under the leakage-safe protocol. If the target is not reached, report the highest valid score, confidence interval, weak endpoints, KG coverage, fold/seed variance and evidence-based next steps. Never manufacture or tune toward the final test number.